# Dauer nociception simulation
- Refer to __ Pechuck et. al. 2022<br>
- 230724 new ID ver.

## Searching valid parameter sets for adult & dauer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tif
import networkx as nx
import os
import copy
import pickle
import itertools
import scipy

from tqdm import tqdm
import scipy.stats as stats
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter
from scipy.stats import linregress
import matplotlib.colors as colors
from matplotlib import cm

## 1) ID and types

In [ ]:
# common type

com_fname = r'your_directory\230724_neurontype_new.csv'    # new common id (230724 -)
com_id_df = pd.read_csv(com_fname)
com_id = np.array(com_id_df[['Neuron','Common_id']])
    # important!!!
    # Cook vs. Witvliet
com_type = np.array(com_id_df['Cook_typecode'])    # change


In [ ]:
# get index (230724 ver.)
# ========================================================================================================================
com_L_idx = [0,2,4,6,8,10,12,14,16,18,20,22,23,25,27,28,30,32,34,36,38,40,42,44,46,48,50,52,54,56,58,59,60,62,64,66,68,70,\
             72,74,75,76,78,80,82,84,86,88,90,92,94,96,98,100,102,104,106,108,109,110,112,114,116,117,119,121,122,124,126,\
             127,128,130,132,134,136,137,139,140,142,144,146,148,150,152,154,156,158,160,162,164,166,168,170,172,174,176,\
             178,180,181,182,183,184,185,186,187,196,197,198,199,200,201,202,203,212,214,216,218,220]
com_R_idx = [1,3,5,7,9,11,13,15,17,19,21,22,24,26,27,29,31,33,35,37,39,41,43,45,47,49,51,53,55,57,58,59,61,63,65,67,69,71,\
             73,74,75,77,79,81,83,85,87,89,91,93,95,97,99,101,103,105,107,108,109,111,113,115,116,118,120,121,123,125,126,\
             127,129,131,133,135,136,138,139,141,143,145,147,149,151,153,155,157,159,161,163,165,167,169,171,173,175,177,\
             179,188,189,190,191,192,193,194,195,204,205,206,207,208,209,210,211,213,215,217,219,221]

com_D_idx = [22,27,58,59,70,71,74,75,80,81,86,87,94,95,108,109,116,121,126,127,130,131,136,146,147,152,153,\
             156,157,160,161,164,165,168,169,176,177,180,181,182,183,184,185,186,187,188,189,190,191,192,\
             193,194,195,212,213,216,217]
com_V_idx = [22,27,58,59,72,73,74,75,84,85,90,91,96,97,108,109,116,121,126,127,134,135,139,148,149,154,155,\
             158,159,162,163,166,167,170,171,178,179,196,197,198,199,200,201,202,203,204,205,206,207,208,\
             209,210,211,214,215,220,221]
# ========================================================================================================================
com_L_idx = np.array(com_L_idx); com_R_idx = np.array(com_R_idx)
com_D_idx = np.array(com_D_idx); com_V_idx = np.array(com_V_idx)

# L/R/D/V id
LR_id = ['ADA','ADE','ADF','ADL','AFD','AIA','AIB','AIM','AIN','AIY','AIZ','ALA','ALM','ALN','AQR','ASE','ASG',\
         'ASH','ASI','ASJ','ASK','AUA','AVA','AVB','AVD','AVE','AVF','AVH','AVJ','AVK','AVL','AVM','AWA','AWB',\
         'AWC','BAG','BDU','CEPD','CEPV','DVA','DVC','FLP','HSN','IL1D','IL1','IL1V','IL2D','IL2','IL2V','OLL',\
         'OLQD','OLQV','PLN','PVC','PVN','PVP','PVQ','PVR','PVT','RIA','RIB','RIC','RID','RIF','RIG','RIH',\
         'RIM','RIP','RIR','RIS','RIV','RMDD','RMD','RMDV','RMED','RME','RMEV','RMF','RMG','RMH','SAAD','SAAV',\
         'SDQ','SIAD','SIAV','SIBD','SIBV','SMBD','SMBV','SMDD','SMDV','URAD','URAV','URB','URX','URYD','URYV']

DV_id = ['ALA','AQR','AVL','AVM','CEP L','CEP R','DVA','DVC','IL1 L','IL1 R','IL2 L','IL2 R','OLQ L','OLQ R',\
         'PVR','PVT','RID','RIH','RIR','RIS','RMD L','RMD R','RME','SAA L','SAA R','SIA L','SIA R',\
         'SIB L','SIB R','SMB L','SMB R','SMD L','SMD R','URA L','URA R','URY L','URY R']


## 2) Connectivity matrices

In [ ]:
# normalized matrix
    # from matrix by HSY (230724 compilation ver.)

norm180_comp = np.load(r"your_directory\norm180_11_compilation_230724.npy", allow_pickle=True); print(norm180_comp.shape)
    # fixed 230724
norm222_comp = np.load(r"your_directory\norm222_11_compilation_230724.npy", allow_pickle=True); print(norm222_comp.shape,"\n")
    # fixed 230724

# binarize
norm180_comp_bi = np.copy(norm180_comp); norm180_comp_bi[norm180_comp_bi!=0] = 1
norm222_comp_bi = np.copy(norm222_comp); norm222_comp_bi[norm222_comp_bi!=0] = 1


# others (original) - 230724 ver. (gap junctions debugged on 240413)
n2u_chem_mat = np.load(r"your_directory\n2u_chem_common_230724.npy", allow_pickle=True); print(n2u_chem_mat.shape)
male_chem_mat = np.load(r"your_directory\male_chem_common_230724.npy", allow_pickle=True); print(male_chem_mat.shape)
dauer_chem_mat = np.load(r"your_directory\dauer_chem_common_230724.npy", allow_pickle=True); print(dauer_chem_mat.shape)

n2u_gap_mat = np.load(r"your_directory\n2u_gap_ori_180_230724.npy", allow_pickle=True); print(n2u_gap_mat.shape)
male_gap_mat = np.load(r"your_directory\male_gap_common_230724.npy", allow_pickle=True); print(male_gap_mat.shape)
dauer_gap_mat = np.load(r"your_directory\dauer1_gap_ori_180_230724.npy", allow_pickle=True); print(dauer_gap_mat.shape)


In [ ]:
# extract submatrix and apply 'Methods' from Pechuck 2022

# submatrix
noci_idx = [32,33, 42,43, 44,45, 46,47, 100,101]  # 230724 ver.
    # target neurons; ASH, AVA, AVB, AVD, PVC (LR pair)

n2u_noci_chem = np.copy(n2u_chem_mat[np.ix_(noci_idx,noci_idx)])
n2u_noci_gap = np.copy(n2u_gap_mat[np.ix_(noci_idx,noci_idx)])
male_noci_chem = np.copy(male_chem_mat[np.ix_(noci_idx,noci_idx)])
male_noci_gap = np.copy(male_gap_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_chem = np.copy(dauer_chem_mat[np.ix_(noci_idx,noci_idx)])
dauer_noci_gap = np.copy(dauer_gap_mat[np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge = np.zeros((5,5)); n2u_noci_gap_merge = np.zeros((5,5))  # change size
male_noci_chem_merge = np.zeros((5,5)); male_noci_gap_merge = np.zeros((5,5))  # change size
dauer_noci_chem_merge = np.zeros((5,5)); dauer_noci_gap_merge = np.zeros((5,5))  # change size


In [ ]:
# chemical
for ii in range(n2u_noci_chem_merge.shape[0]):
    for jj in range(n2u_noci_chem_merge.shape[1]):
        square = n2u_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2(or 4?) vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_noci_chem_merge[ii,jj] = square_avg

for ii in range(male_noci_chem_merge.shape[0]):
    for jj in range(male_noci_chem_merge.shape[1]):
        square = male_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_chem_merge[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge.shape[0]):
    for jj in range(dauer_noci_chem_merge.shape[1]):
        square = dauer_noci_chem[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge[ii,jj] = square_avg


# gap junction
for ii in range(n2u_noci_gap_merge.shape[0]):
    for jj in range(n2u_noci_gap_merge.shape[1]):
        square = n2u_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            n2u_noci_gap_merge[ii,jj] = square_avg

for ii in range(male_noci_gap_merge.shape[0]):
    for jj in range(male_noci_gap_merge.shape[1]):
        square = male_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            male_noci_gap_merge[ii,jj] = square_avg

for ii in range(dauer_noci_gap_merge.shape[0]):
    for jj in range(dauer_noci_gap_merge.shape[1]):
        square = dauer_noci_gap[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
            # options for denominator = 2 vs. np.nonzero(square)[0].size
        if ii==jj:
            continue
        else:
            dauer_noci_gap_merge[ii,jj] = square_avg


In [ ]:
# check full
print_df = pd.DataFrame(data=dauer_noci_chem,\
                        index=[com_id[noci_idx,0]], columns=[com_id[noci_idx,0]])
print_df

In [ ]:
# check merged
print_df = pd.DataFrame(data=dauer_noci_chem_merge,\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

### Normalize according to 'norm180_comp' ver.

In [ ]:
# from normalized matrix
n2u_noci_chem_norm = np.copy(norm180_comp[:,:,9][np.ix_(noci_idx,noci_idx)])
dauer_noci_chem_norm = np.copy(norm180_comp[:,:,10][np.ix_(noci_idx,noci_idx)])

# LR pair merge
n2u_noci_chem_merge_norm = np.zeros((5,5))
dauer_noci_chem_merge_norm = np.zeros((5,5))

# chemical
for ii in range(n2u_noci_chem_merge_norm.shape[0]):
    for jj in range(n2u_noci_chem_merge_norm.shape[1]):
        square = n2u_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            n2u_noci_chem_merge_norm[ii,jj] = square_avg

for ii in range(dauer_noci_chem_merge_norm.shape[0]):
    for jj in range(dauer_noci_chem_merge_norm.shape[1]):
        square = dauer_noci_chem_norm[2*ii:2*(ii+1),2*jj:2*(jj+1)]
        square_avg = np.sum(square) / 4
        if ii==jj:
            continue
        else:
            dauer_noci_chem_merge_norm[ii,jj] = square_avg


In [ ]:
# reshape dauer nociception circuit into Pechuk order

new_order = [0,1,3,2,4]

# adjust strength
mult_norm = np.ones((5,5))  # factor to multiply to dauer normalized matrix

ori_mat_temp = np.copy(n2u_noci_chem_merge[np.ix_(new_order,new_order)])
norm_mat_temp = np.copy(n2u_noci_chem_merge_norm[np.ix_(new_order,new_order)])
for ii in range(mult_norm.shape[0]):
    for jj in range(mult_norm.shape[1]):
        if norm_mat_temp[ii,jj]==0:
            continue
        else:
            mult_norm[ii,jj] = ori_mat_temp[ii,jj] / norm_mat_temp[ii,jj]

print(mult_norm)  # find the most frequent value (except 1)


In [ ]:
# check dataframe
print("  N2U (Cook)")
print_df = pd.DataFrame(data=np.round(ori_mat_temp,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

In [ ]:
# check dataframe
print("  N2U (norm.)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp,3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

In [ ]:
# check dataframe
print("  N2U (scaled)")
print_df = pd.DataFrame(data=np.round(norm_mat_temp * mult_norm[0,1],3),\
                        index=["ASH","AVA","AVB","AVD","PVC"],\
                        columns=["ASH","AVA","AVB","AVD","PVC"])
print_df

---

In [ ]:
import extract_connectivity_df as ecd
import supporting_functions as sf

path_to_cook_file = r"your_directorySI_5_cook_original.xlsx"

herm_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite chemical", header = None)
herm_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite gap jn symmetric", header = None)
male_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male chemical", header = None)
male_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male gap jn symmetric", header = None)

# =============================================================================
# Define neurons of interest:
# In circuit_neurons the keys are the name of the neural class, and the values are
# the regex expressions to find the cells in Cook's data.  
# =============================================================================
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]',\
                   'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

Sgap_herm_head,Schem_herm_head,Sgap_male_head,Schem_male_head = ecd.final_connectivity_matrix(herm_chem_full_connectivity,\
                                                                                              herm_gapj_full_connectivity,\
                                                                                              male_chem_full_connectivity,\
                                                                                              male_gapj_full_connectivity,\
                                                                                              circuit_neurons)


In [ ]:
# reshape dauer nociception circuit into Pechuk order
    # Original order = ASH, AVA, AVB, AVD, PVC
    #   Pechuk order = ASH, AVA, AVD, AVB, PVC
    # assume that connectivity to A and B is similar to hermaphrodite (i.e., use hermaphrodite's data)

Schem_dauer_head = np.zeros((7,7))  # mind the direction (transpose)
Sgap_dauer_head = np.zeros((7,7))  # mind the direction (transpose)

# chemical
dauer_chem_adjusted = np.copy( (dauer_noci_chem_merge_norm[np.ix_(new_order,new_order)] * mult_norm[0,1]) )  # Pechuk order
Schem_dauer_head[:5,:5] = np.copy(dauer_chem_adjusted.T)  # transpose
Schem_dauer_head[5:,:] = np.copy(Schem_herm_head[5:,:])  # A, B row (from herm.)
Schem_dauer_head[:,5:] = np.copy(Schem_herm_head[:,5:])  # A, B column (from herm.)

# electrical (don't need to adjust strength; strength by section number)
dauer_gap_adjusted = np.copy( dauer_noci_gap_merge[np.ix_(new_order,new_order)] )  # Pechuk order
Sgap_dauer_head[:5,:5] = np.copy(dauer_gap_adjusted.T)
Sgap_dauer_head[5:,:] = np.copy(Sgap_herm_head[5:,:])
Sgap_dauer_head[:,5:] = np.copy(Sgap_herm_head[:,5:])


In [ ]:
# check
print("Dauer")
print_df_1 = pd.DataFrame(data=np.round(Sgap_dauer_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_1

In [ ]:
# check
print("Herm.")
print_df_2 = pd.DataFrame(data=np.round(Sgap_herm_head[:5,:5].T,2),\
                        index=["ASH","AVA","AVD","AVB","PVC"],\
                        columns=["ASH","AVA","AVD","AVB","PVC"])  # Schem vs. Sgap
print_df_2

# Simulation application
> original code from *Pechuk et al 2022*

In [ ]:
from itertools import product

# =============================================================================
# Parameters' combinations
# From each parameter 7 values were sampled, equally distributed on a logarithmic scale.
# =============================================================================
resistance_per_area = np.geomspace(1500 * 10 **3, 15 * 10**3, 7)
capacitance = np.geomspace(10 **-7 , 10**-5, 7)
gchem = np.geomspace(10**-9, 10**-12, 7)
ggap = np.geomspace(10**-9, 10**-12, 7)
AVB_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
AVA_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
PVC_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
[RES, CAP, GCHEM, GGAP, AVB_DV, AVA_DV, PVC_DV]= np.meshgrid(resistance_per_area,\
                                                             capacitance, gchem, ggap, AVB_dV, AVA_dV, PVC_dV)
len_df = len(resistance_per_area) * len(capacitance) * len(gchem) * len(ggap) *len(AVB_dV) *len(AVA_dV) *len(PVC_dV)
dict_for_df = {"Rm": np.reshape(RES, len_df), "Capacitance" : np.reshape(CAP, len_df),
               "Chemical_Conductance" : np.reshape(GCHEM, len_df), "Gap_Conductance" : np.reshape(GGAP, len_df),
               "Voltage_Change_AVB": np.reshape(AVB_DV, len_df), "Voltage_Change_AVA": np.reshape(AVA_DV, len_df),
               "Voltage_Change_PVC": np.reshape(PVC_DV, len_df)}
Parameters_df = pd.DataFrame(dict_for_df)

# fixed parameters:
surace_area = 15 * 10**(-6)  # units: cm2
Rin = Parameters_df.Rm.map(lambda x: x/surace_area)
Parameters_df["Rin"] = Rin

# input that would trigger 100mV change:
    # V = IR
    # 75 * 10**-3 = I * Rin
sensory_I = Parameters_df.Rin.map(lambda x: 75 * 10**-3/x)
Parameters_df["Sensory_Input"] = sensory_I  # simulate all of varying sensory inputs

# input to the interneurons: 
I_AVB = Parameters_df.Voltage_Change_AVB/Parameters_df.Rin
I_AVA = Parameters_df.Voltage_Change_AVA/Parameters_df.Rin
I_PVC = Parameters_df.Voltage_Change_PVC/Parameters_df.Rin

Parameters_df["I_AVB"] = I_AVB
Parameters_df["I_AVA"] = I_AVA
Parameters_df["I_PVC"] = I_PVC


In [ ]:
import extract_connectivity_df as ecd
import supporting_functions as sf

# =============================================================================
# Define neurons of interest:
# In circuit_neurons the keys are the name of the neural class, and the values are
# the regex expressions to find the cells in Cook's data.  
# =============================================================================
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]',\
                   'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# =============================================================================
# Find all combinations of polarity and parameters
    # Parameters_df, possible_combinations_df -> already made from saved files
# =============================================================================
parameters_pol_df = pd.DataFrame()
parameters_num = len(Parameters_df)  # 823,543
polarity_num = len(possible_combinations_df)  # 16 (2^4)
parameters_pol_df["parameters_index"] = Parameters_df.index.tolist()
parameters_pol_df = pd.concat([parameters_pol_df] * polarity_num, ignore_index=True)  # 117,649 * 16
parameters_pol_df["polarity_index"] = np.repeat(np.arange(0, polarity_num ,1), parameters_num)

parameters_pol_df["parameters_index"] = parameters_pol_df["parameters_index"].astype("int32")
parameters_pol_df["polarity_index"] = parameters_pol_df["polarity_index"].astype("int8")
print("\n'parameters_pol_df' made. \n Shape :", parameters_pol_df.shape)

# =============================================================================
# setting the enviroment
# =============================================================================
# time:
t = np.linspace(0,15,75001)  # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1))  #units: V


#### Functions for plot

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

def plot_reverse_probability_per_input_dauer(sensory_voltage_changes, responses_df_dauer_and_herm, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_dauer_and_herm - df with the WT responses of both stages
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    mean_reversal_herm = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_dauer_and_herm["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_dauer_and_herm["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='darkturquoise')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 16)
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
    if len(args)>0:
        plt.legend(labels + ["Wild-type hermaphrodites","Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type hermaphrodites","Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
    plt.title(title_on_plot, fontsize = 18)
    plt.savefig(title +".png", dpi=400, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

def plot_reverse_probability_per_input_all(sensory_voltage_changes, responses_df_all, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    mean_reversal_herm = []
    mean_reversal_male = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_male.append(np.mean(responses_df_all["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_all["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown')  # orangish
    plt.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = 'darkturquoise')  # bluish
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato')  # redish
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 16)
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
    if len(args)>0:
        plt.legend(labels + ["Wild-type hermaphrodites","Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='upper right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type hermaphrodites","Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='upper right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
    plt.title(title_on_plot, fontsize = 18)
    plt.savefig(title +".png", dpi=400, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

def plot_reverse_probability_per_input_AD(sensory_voltage_changes, responses_df_all, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    mean_reversal_herm = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_all["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = 'sandybrown')  # orangish
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='tomato')  # redish
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 16)
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=16)
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
    plt.title(title_on_plot, fontsize = 18)
    plt.savefig(title +".png", dpi=400, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

    # manuscript ver.
    # svg ; change color ; change font


def plot_reverse_probability_per_input_AD_svg(sensory_voltage_changes, responses_df_all, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))
    
    mean_reversal_herm = []
    mean_reversal_dauer = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_all["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = '#1F77B4')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='#FF7F0E')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 14, fontname='sans-serif')
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=14, fontname='sans-serif')
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
#     plt.title(title_on_plot, fontsize = 18, fontname='sans-serif')
    plt.savefig(title +".svg", dpi=300, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

    # manuscript ver.
    # svg ; change color ; change font


def plot_reverse_probability_per_input_AD_svg_two(sensory_voltage_changes, responses_df_all1,responses_df_all2, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))
    
    mean_reversal_herm = []
    mean_reversal_dauer1 = []
    mean_reversal_dauer2 = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all1["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer1.append(np.mean(responses_df_all1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer2.append(np.mean(responses_df_all2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = '#1F77B4')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer1, marker = "o", label = "Dauers", color ='#FF7F0E')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer2, marker = "o", label = "Dauers", color ='#D62728')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 14, fontname='sans-serif')
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=14, fontname='sans-serif')
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type dauers", "Wild-type dauers (Rm full)"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type dauers", "Wild-type dauers (Rm full)"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
#     plt.title(title_on_plot, fontsize = 18, fontname='sans-serif')
    plt.savefig(title +".svg", dpi=300, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

    # manuscript ver.
    # svg ; change color ; change font


def plot_reverse_probability_per_input_AD_svg_two2(sensory_voltage_changes, responses_df_all1,responses_df_all2, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))
    
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    mean_reversal_herm = []
    mean_reversal_dauer1 = []
    mean_reversal_dauer2 = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all1["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer2.append(np.mean(responses_df_all2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = '#1F77B4')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer2, marker = "o", label = "Dauers", color ='#FF7F0E')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 14, fontname='sans-serif')
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=14, fontname='sans-serif')
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
#     plt.title(title_on_plot, fontsize = 18, fontname='sans-serif')
    plt.savefig(title +".svg", dpi=300, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

    # manuscript ver.
    # svg ; change color ; change font


def plot_reverse_probability_per_input_AD_svg_two3(sensory_voltage_changes, responses_df_all1,responses_df_all2, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))
    
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    mean_reversal_herm = []
    mean_reversal_male = []
    mean_reversal_dauer1 = []
    mean_reversal_dauer2 = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all1["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_male.append(np.mean(responses_df_all1["male_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer2.append(np.mean(responses_df_all2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2    
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = '#1F77B4')
    plt.plot(sensory_voltage_changes,mean_reversal_male, marker = "o", label = "Males", color = '#2CA02C')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer2, marker = "o", label = "Dauers", color ='#FF7F0E')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 14, fontname='sans-serif')
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=14, fontname='sans-serif')
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type males", "Wild-type dauers"] , fontsize = 14, loc='lower right', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
#     plt.title(title_on_plot, fontsize = 18, fontname='sans-serif')
    plt.savefig(title +".svg", dpi=300, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

---

## Hermaphrodite , Male , Dauer

- Copy 667 set data from `herm_cell_activity_basic_mutual.csv` from Pechuk et al. <br><br>
- Rearrange file so that it can be used to get `behavioral conditions` in dauer network as well. <br><br>
- Check if there are sets that meet all hermaphrodite `adult`, `male`, and `dauer` condition.

In [ ]:
# =============================================================================
mutual_sets = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual.csv")
mutual_sets_final = mutual_sets[mutual_sets["polarity_index"] == 8].reset_index()  # polarity 8 (667 rows)

Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk

pol_fname = r"your_directory\Polarity_combinations_sample.pkl"
possible_combinations_df = pd.read_pickle(pol_fname)


# just copy colums 'index', 'parameters_index', and 'polarity_index'
    # run again with this dataframe
df_temp = pd.DataFrame()
df_temp.loc[:, 'index'] = mutual_sets_final.loc[:, 'index']
df_temp.loc[:, 'parameters_index'] = mutual_sets_final.loc[:, 'parameters_index']
df_temp.loc[:, 'polarity_index'] = mutual_sets_final.loc[:, 'polarity_index']
print(df_temp.head(5))
# =============================================================================

In [ ]:
# df_temp.to_csv(r"your_directory\valid_667_initial_df.csv")

In [ ]:
# # =============================================================================
# # Set environment
# # =============================================================================
# # define neurons of interest:
# circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
# neurons = list(circuit_neurons.keys())

# # time:
# t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
# correction_to_ms = int(round(0.001/(t[1]-t[0])))
# sensory_start = 5000 * correction_to_ms
# sensory_end = 10000 * correction_to_ms

# # resting state: 
# Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# # =============================================================================
# # Input that would trigger a specific voltage change:
# # =============================================================================
# # V = IR, I = V/R
# sensory_input_calc = lambda res, volt: volt/res
# sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]

In [ ]:
# running 667 set to check basic condition of dauer


Parameters_Job = df_temp.copy(); print(Parameters_Job.shape)

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
male_basic_conditions = []; male_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
Sgap_male_adj = []; Schem_male_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head,[1.2, 1, 0.8], "chem")
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head,[1.2, 1, 0.8], "gap")
    Schem_male_row, mult_chem_male = ecd.adjusting_strengths(Schem_male_head,[1.2, 1, 0.8], "chem")
    Sgap_male_row, mult_gap_male =  ecd.adjusting_strengths(Sgap_male_head,[1.2, 1, 0.8], "gap")
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_head,[1.2, 1, 0.8], "chem")
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_head,[1.2, 1, 0.8], "gap")

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_male_row = sf.adjust_polarity(Schem_male_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_male_adj.append(Sgap_male_row)
    Schem_male_adj.append(Schem_male_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Males
# =============================================================================
    sex = "Males"
    V_m_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male_row, Sgap_male_row, sex)
    basic_conditions_male = sf.basic_conditions_check(V_m_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    male_basic_conditions.append(basic_conditions_male)
    if basic_conditions_male == (True):
        male_cell_act_conditions.append(sf.cells_activity_conditions_check(V_m_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        male_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer_row, Sgap_dauer_row, sex)  # changed Schem/Sgap matrices
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["male_basic_conditions"] = male_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["male_cell_act_conditions"] = male_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_male_adj"] = Schem_male_adj
Parameters_Job["Sgap_male_adj"] = Sgap_male_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_male"] = ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["male_cell_act_conditions"] == True) | (Parameters_Job["male_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))
    # mutual = herm & male & dauer


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_667_250716.csv")  # change name


In [ ]:
# select mutual parameter sets (hermaphrodite, male, dauer)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_667_250716.csv", index_col=0)  # check name
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["male_basic_conditions"] == "True") | (single_run_df["male_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 
    # herm, male, dauer

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_herm_male_dauer_667_250716.csv")  # check name
herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()
herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_667_250716.csv")  # check name
# =============================================================================


In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_667_250716.csv"  # change name
herm_cell_activity_basic_mutual_667 = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df.csv"  # original
Parameters_df = pd.read_csv(fname2, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]

Different_stimulus_Job = herm_cell_activity_basic_mutual_667.copy()  # check name


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons))) 
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons)))
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row], (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))    
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        
# =============================================================================
# Hermaphrodites
# =============================================================================
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        herm_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Males
# =============================================================================
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        male_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["male_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["male_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(male_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_250716.csv")

In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_all(sensory_voltage_changes,\
                                       different_stimuli_samp_pol,\
                                       r"your_directory\predicted_avoidance_adult_male_dauer_667_260102",\
                                       (1.02,-0.18))  # modified function


In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD(sensory_voltage_changes,
                                      different_stimuli_samp_pol,
                                      r"your_directory\predicted_avoidance_adult_dauer_667_260102",
                                      (1.0,0.0))  # modified function


In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg(sensory_voltage_changes,
                                          different_stimuli_samp_pol,
                                          r"your_directory\predicted_avoidance_adult_dauer_667_260131",
                                          (1.0,0.0))  # modified function


## 667 parameter x Rm full range simulation

- Want to check the effect of Rm in dauer. <br><br>
- 667 combination X All Rm range (7 values) combination.

In [ ]:
# =============================================================================
mutual_sets = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual.csv")
mutual_sets_final = mutual_sets[mutual_sets["polarity_index"] == 8].reset_index()  # polarity 8 (667 rows)

Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk

pol_fname = r"your_directory\Polarity_combinations_sample.pkl"
possible_combinations_df = pd.read_pickle(pol_fname)


# just copy colums 'index', 'parameters_index', and 'polarity_index'
    # run again with this dataframe
df_temp = pd.DataFrame()
df_temp.loc[:, 'index'] = mutual_sets_final.loc[:, 'index']
df_temp.loc[:, 'parameters_index'] = mutual_sets_final.loc[:, 'parameters_index']
df_temp.loc[:, 'polarity_index'] = mutual_sets_final.loc[:, 'polarity_index']
print(df_temp.head(5))
# =============================================================================

---
1. use 'parameters_index' column in 'herm_cell_activity_basic_mutual.csv' <br><br>
2. extract parameter sets of those indeces in 'Parameters_df' (823,542 all possible combinations in this file) <br><br>
3. get data on parameter sets except Rm <br><br>
4. make new combination with Rm-excluded set X all Rm range -> theoretically 667 X 7 = 4,669 combination <br><br>
5. get indeces of 4,669 sets and make a new df to run simulation
---

In [ ]:
# originale parameters (667)

ori_idx = list(mutual_sets_final['parameters_index'])  # list of indices

ori_param_list = Parameters_df.iloc[ori_idx]
print(ori_param_list.shape)

In [ ]:
ori_param_list.head(5)

In [ ]:
ori_param_temp = ori_param_list.iloc[:,1:].copy()  # no 'Rm'

print(ori_param_temp.shape)
ori_param_temp.head(5)

In [ ]:
# expand dataframe

ori_param_ext = ori_param_temp.copy()

for ii in range(6):
    ori_param_ext = pd.concat([ori_param_ext, ori_param_temp], ignore_index=True)

print(ori_param_ext.shape)

In [ ]:
ori_param_ext.head(5)

resistance_per_area = np.geomspace(1500 * 10 ^3, 15 * 10^3, 7) <br><br>
capacitance = np.geomspace(10 ^-7 , 10^-5, 7) <br><br>
gchem = np.geomspace(10^-9, 10^-12, 7) <br><br>
ggap = np.geomspace(10^-9, 10^-12, 7) <br><br>
AVB_dV = np.geomspace(75 * 10^-3,  1* 10^-3, 7) <br><br>
AVA_dV = np.geomspace(75 * 10^-3,  1* 10^-3, 7) <br><br>
PVC_dV = np.geomspace(75 * 10^-3,  1* 10^-3, 7)

In [ ]:
# Add Rm column

rm_val = np.geomspace(1500 * 10 **3, 15 * 10**3, 7)

rm_val_ext = []
for ii in range(rm_val.size):
    rm_val_ext = rm_val_ext + list([rm_val[ii]])*667

print(len(rm_val_ext))


In [ ]:
# insert 'Rm' column

ori_param_ext.insert(loc=0, column='Rm', value=rm_val_ext)

In [ ]:
print(ori_param_ext.shape)
ori_param_ext.head(5)

In [ ]:
# Recalculate 'Rin' and 'sensory input'
    # Note that Rin is calculated from Rm


# fixed parameters:
surace_area = 15 * 10**(-6)

Rin_temp = ori_param_ext.Rm.map(lambda x: x/surace_area)
ori_param_ext["Rin"] = Rin_temp

# input that would trigger 100mV change:
    # V = IR
    # 75 * 10**-3 = I * Rin
sensory_I_temp = ori_param_ext.Rin.map(lambda x: 75 * 10**-3/x)
ori_param_ext["Sensory_Input"] = sensory_I_temp  # simulate all of varying sensory inputs


In [ ]:
print(ori_param_ext.shape)
ori_param_ext.head(5)

In [ ]:
Parameters_df.head(5)

In [ ]:
# get indeces from 'Parameters_df'
    # there might be duplicates

ext_idx = []

DF1_np = Parameters_df.to_numpy()
DF2_np = ori_param_ext.to_numpy()


for ii in tqdm(range(DF2_np.shape[0])):
    ROW = DF2_np[ii,:7]  # for 7 parameters
    idx_cnt=0
    for jj in range(DF1_np.shape[0]):
        if np.array_equal(ROW,DF1_np[jj,:7])==True:
            ext_idx.append(jj)
            idx_cnt+=1
        else:
            continue
    if idx_cnt!=1:
        print("Check ", ii)
    else:
        continue


#     diff_np = DF1_np[:,:7] - ROW  # 823,543 X 7
#     diff_sum = np.sum(diff_np, axis=1)  # 823,543 X 1
    
#     match_idx = np.where(diff_sum==0)[0]
#     if match_idx.size!=1:
#         print(match_idx.size)
#     else:
#         ext_idx.append(match_idx[0])

print(len(ext_idx))


In [ ]:
ext_idx_set = set(ext_idx)

ext_idx_final = list(ext_idx_set)
ext_idx_final.sort()  # ascending order

print(len(ext_idx_final))

In [ ]:
# just copy colums 'parameters_index', and 'polarity_index'
    # run again with this dataframe
df_temp = pd.DataFrame()
df_temp.loc[:, 'parameters_index'] = ext_idx_final
df_temp.loc[:, 'polarity_index'] = 8
print(df_temp.head(5))

In [ ]:
check_isin = np.isin(ori_idx, ext_idx_final)

print(check_isin.shape)
print(np.sum(check_isin))

In [ ]:
# setting the enviroment

# time:
t = np.linspace(0,15,75001)  # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1))  #units: V


In [ ]:
# running 4,298 sets to check expanded Rm in dauer
    # just on dauer circuit
    # herm and male would not fit anyway (except original 667)


Parameters_Job = df_temp.copy(); print(Parameters_Job.shape)

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
male_basic_conditions = []; male_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
Sgap_male_adj = []; Schem_male_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head,[1.2, 1, 0.8], "chem")
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head,[1.2, 1, 0.8], "gap")
    Schem_male_row, mult_chem_male = ecd.adjusting_strengths(Schem_male_head,[1.2, 1, 0.8], "chem")
    Sgap_male_row, mult_gap_male =  ecd.adjusting_strengths(Sgap_male_head,[1.2, 1, 0.8], "gap")
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_head,[1.2, 1, 0.8], "chem")
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_head,[1.2, 1, 0.8], "gap")

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_male_row = sf.adjust_polarity(Schem_male_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_male_adj.append(Sgap_male_row)
    Schem_male_adj.append(Schem_male_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Males
# =============================================================================
    sex = "Males"
    V_m_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male_row, Sgap_male_row, sex)
    basic_conditions_male = sf.basic_conditions_check(V_m_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    male_basic_conditions.append(basic_conditions_male)
    if basic_conditions_male == (True):
        male_cell_act_conditions.append(sf.cells_activity_conditions_check(V_m_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        male_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer_row, Sgap_dauer_row, sex)  # changed Schem/Sgap matrices
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["male_basic_conditions"] = male_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["male_cell_act_conditions"] = male_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_male_adj"] = Schem_male_adj
Parameters_Job["Sgap_male_adj"] = Sgap_male_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_male"] = ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["male_cell_act_conditions"] == True) | (Parameters_Job["male_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))
    # mutual = herm & male & dauer


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_dauer_Rm_ext_4298_260108.csv")  # change name


In [ ]:
# select mutual parameter sets (but this time it's dauer-only)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_dauer_Rm_ext_4298_260108.csv", index_col=0)  # check name
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")

some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["male_basic_conditions"] == "True") | (single_run_df["male_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_dauer_Rm_ext_4298_260112.csv")  # check name

herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()

herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260112.csv")  # check name
# =============================================================================


260109 : 1527 sets<br>
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 

herm_cell_activity_basic_mutual = parameters_pol_samp_df[((parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True) | (parameters_pol_samp_df["dauer_basic_conditions"] == "True") | (parameters_pol_samp_df["dauer_basic_conditions"] == True))].reset_index()

260112 : <br>
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["male_basic_conditions"] == "True") | (single_run_df["male_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 

herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()

In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260112.csv"  # change name
herm_cell_activity_basic_mutual_Rm_ext = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df.csv"  # original
Parameters_df = pd.read_csv(fname2, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]

Different_stimulus_Job = herm_cell_activity_basic_mutual_Rm_ext.copy()  # check name !!!


# =============================================================================
# Initializing response arrays
# =============================================================================
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))    
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_dauer, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A


# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


In [ ]:
for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260112.csv")

---

## Surface area adjustment for dauer (+ Rm ext.)

- Adult : 15 x 10^(-6)   -->   Dauer : 12 x 10^(-6)

In [ ]:
# # 'Parameters_df' for dauer (surface area adjusted)


# from itertools import product

# # =============================================================================
# # Parameters' combinations
# # From each parameter 7 values were sampled, equally distributed on a logarithmic scale.
# # =============================================================================
# resistance_per_area = np.geomspace(1500 * 10 **3, 15 * 10**3, 7)
# capacitance = np.geomspace(10 **-7 , 10**-5, 7)
# gchem = np.geomspace(10**-9, 10**-12, 7)
# ggap = np.geomspace(10**-9, 10**-12, 7)
# AVB_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
# AVA_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
# PVC_dV = np.geomspace(75 * 10**-3,  1* 10**-3, 7)
# [RES, CAP, GCHEM, GGAP, AVB_DV, AVA_DV, PVC_DV]= np.meshgrid(resistance_per_area,\
#                                                              capacitance, gchem, ggap, AVB_dV, AVA_dV, PVC_dV)
# len_df = len(resistance_per_area) * len(capacitance) * len(gchem) * len(ggap) *len(AVB_dV) *len(AVA_dV) *len(PVC_dV)
# dict_for_df = {"Rm": np.reshape(RES, len_df), "Capacitance" : np.reshape(CAP, len_df),
#                "Chemical_Conductance" : np.reshape(GCHEM, len_df), "Gap_Conductance" : np.reshape(GGAP, len_df),
#                "Voltage_Change_AVB": np.reshape(AVB_DV, len_df), "Voltage_Change_AVA": np.reshape(AVA_DV, len_df),
#                "Voltage_Change_PVC": np.reshape(PVC_DV, len_df)}


# Parameters_df_D = pd.DataFrame(dict_for_df)  # '_D' for dauer

# # fixed parameters:
# surace_area_D = 12 * 10**(-6)
#     # units: cm2 (Changed this value !!!)


# Rin_D = Parameters_df_D.Rm.map(lambda x: x/surace_area_D)
# Parameters_df_D["Rin"] = Rin_D

# # input that would trigger 100mV change:
#     # V = IR
#     # 75 * 10**-3 = I * Rin
# sensory_I_D = Parameters_df_D.Rin.map(lambda x: 75 * 10**-3/x)
# Parameters_df_D["Sensory_Input"] = sensory_I_D  # simulate all of varying sensory inputs

# # input to the interneurons: 
# I_AVB_D = Parameters_df_D.Voltage_Change_AVB/Parameters_df_D.Rin
# I_AVA_D = Parameters_df_D.Voltage_Change_AVA/Parameters_df_D.Rin
# I_PVC_D = Parameters_df_D.Voltage_Change_PVC/Parameters_df_D.Rin

# Parameters_df_D["I_AVB"] = I_AVB_D
# Parameters_df_D["I_AVA"] = I_AVA_D
# Parameters_df_D["I_PVC"] = I_PVC_D

# Parameters_df_D.to_csv(r"D:\noci_pol8_result\surf_adj\Parameters_combinations_df_D.csv")


In [ ]:
# =============================================================================
mutual_sets = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual.csv")
mutual_sets_final = mutual_sets[mutual_sets["polarity_index"] == 8].reset_index()  # polarity 8 (667 rows)

Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk
Parameters_df_D = pd.read_csv(r"your_directory\Parameters_combinations_df_D.csv", index_col = 0)  # dauer adjusted



pol_fname = r"your_directory\Polarity_combinations_sample.pkl"  # same for both
possible_combinations_df = pd.read_pickle(pol_fname)


# just copy colums 'index', 'parameters_index', and 'polarity_index'
    # run again with this dataframe
df_temp = pd.DataFrame()
df_temp.loc[:, 'index'] = mutual_sets_final.loc[:, 'index']
df_temp.loc[:, 'parameters_index'] = mutual_sets_final.loc[:, 'parameters_index']
df_temp.loc[:, 'polarity_index'] = mutual_sets_final.loc[:, 'polarity_index']
print(df_temp.head(5))
# =============================================================================

### ( Run for 667 sets first )

In [ ]:
# running 667 set to check basic condition of dauer


Parameters_Job = df_temp.copy(); print(Parameters_Job.shape)

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
male_basic_conditions = []; male_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
Sgap_male_adj = []; Schem_male_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head,[1.2, 1, 0.8], "chem")
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head,[1.2, 1, 0.8], "gap")
    Schem_male_row, mult_chem_male = ecd.adjusting_strengths(Schem_male_head,[1.2, 1, 0.8], "chem")
    Sgap_male_row, mult_gap_male =  ecd.adjusting_strengths(Sgap_male_head,[1.2, 1, 0.8], "gap")
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_head,[1.2, 1, 0.8], "chem")
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_head,[1.2, 1, 0.8], "gap")

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_male_row = sf.adjust_polarity(Schem_male_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_male_adj.append(Sgap_male_row)
    Schem_male_adj.append(Schem_male_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A
    I_D = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df_D.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df_D.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df_D.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df_D.I_PVC[parameter_ind], 0) # units: A (for dauer)

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Males
# =============================================================================
    sex = "Males"
    V_m_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male_row, Sgap_male_row, sex)
    basic_conditions_male = sf.basic_conditions_check(V_m_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    male_basic_conditions.append(basic_conditions_male)
    if basic_conditions_male == (True):
        male_cell_act_conditions.append(sf.cells_activity_conditions_check(V_m_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        male_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df_D.Rm[parameter_ind] * Parameters_df_D.Capacitance[parameter_ind],\
                                     Parameters_df_D.Rin[parameter_ind], Parameters_df_D.Chemical_Conductance[parameter_ind],\
                                     Parameters_df_D.Gap_Conductance[parameter_ind], I_D, Schem_dauer_row, Sgap_dauer_row, sex)  # changed Schem/Sgap matrices
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)  # changed to '_D' for dauer
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)  # ( no need to add column for dauer? )
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["male_basic_conditions"] = male_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["male_cell_act_conditions"] = male_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_male_adj"] = Schem_male_adj
Parameters_Job["Sgap_male_adj"] = Sgap_male_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_male"] = ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["male_cell_act_conditions"] == True) | (Parameters_Job["male_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))
    # mutual = herm & male & dauer


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_667_260116.csv")  # change name


In [ ]:
# select mutual parameter sets (hermaphrodite, male, dauer)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_667_260116.csv", index_col=0)  # check name
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["male_basic_conditions"] == "True") | (single_run_df["male_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 
    # herm, male, dauer

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_herm_male_dauer_667_260116.csv")  # check name
herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()
herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_667_260116.csv")  # check name
# =============================================================================


In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_667_260116.csv"  # change name
herm_cell_activity_basic_mutual_surf = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df_D.csv"  # for dauer
Parameters_df_D = pd.read_csv(fname2, index_col = 0)

fname3 = r"your_directory\Parameters_combinations_df.csv"
Parameters_df = pd.read_csv(fname3, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]

Different_stimulus_Job = herm_cell_activity_basic_mutual_surf.copy()  # check name


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons))) 
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons)))
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row], (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))    
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        
        sensory_input_D = sensory_input_calc(Parameters_df_D.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I_D = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input_D,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df_D.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df_D.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df_D.I_PVC[parameter_ind], 0) # units: A (for dauer)
        
# =============================================================================
# Hermaphrodites
# =============================================================================
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        herm_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Males
# =============================================================================
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        male_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df_D.Rm[parameter_ind] * Parameters_df_D.Capacitance[parameter_ind],  Parameters_df_D.Rin[parameter_ind], Parameters_df_D.Chemical_Conductance[parameter_ind], Parameters_df_D.Gap_Conductance[parameter_ind], I_D, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["male_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["male_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(male_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")

In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_all(sensory_voltage_changes,\
                                       different_stimuli_samp_pol,\
                                       r"your_directory\predicted_avoidance_adult_male_dauer_667_260116",\
                                       (1.02,-0.18))  # modified function


In [ ]:
# plot


from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg(sensory_voltage_changes,
                                          different_stimuli_samp_pol,
                                          r"your_directory\predicted_avoidance_adult_male_dauer_667_260202",
                                          (1.0,0.0))  # modified function


### ( Rm ext. )

In [ ]:
# originale parameters (667) - for dauer

ori_idx = list(mutual_sets_final['parameters_index'])  # list of indices

ori_param_list_D = Parameters_df_D.iloc[ori_idx]  # changed this to '_D' for dauer
print(ori_param_list_D.shape)

In [ ]:
ori_param_D_temp = ori_param_list_D.iloc[:,1:].copy()  # no 'Rm'

print(ori_param_D_temp.shape)
ori_param_D_temp.head(5)

In [ ]:
# expand dataframe

ori_param_D_ext = ori_param_D_temp.copy()

for ii in range(6):
    ori_param_D_ext = pd.concat([ori_param_D_ext, ori_param_D_temp], ignore_index=True)

print(ori_param_D_ext.shape)

In [ ]:
# Add Rm column

rm_val = np.geomspace(1500 * 10 **3, 15 * 10**3, 7)

rm_val_ext = []
for ii in range(rm_val.size):
    rm_val_ext = rm_val_ext + list([rm_val[ii]])*667

print(len(rm_val_ext))


In [ ]:
# insert 'Rm' column

ori_param_D_ext.insert(loc=0, column='Rm', value=rm_val_ext)

In [ ]:
print(ori_param_D_ext.shape)
ori_param_D_ext.head(5)

In [ ]:
# Recalculate 'Rin' and 'sensory input'
    # Note that Rin is calculated from Rm


# fixed parameters:
surace_area_D = 12 * 10**(-6)
    # units: cm2 (Changed this value !!!)

Rin_D_temp = ori_param_D_ext.Rm.map(lambda x: x/surace_area_D)
ori_param_D_ext["Rin"] = Rin_D_temp

# input that would trigger 100mV change:
    # V = IR
    # 75 * 10**-3 = I * Rin
sensory_I_D_temp = ori_param_D_ext.Rin.map(lambda x: 75 * 10**-3/x)
ori_param_D_ext["Sensory_Input"] = sensory_I_D_temp  # simulate all of varying sensory inputs


In [ ]:
print(ori_param_D_ext.shape)
ori_param_D_ext.head(5)

In [ ]:
# get indeces from 'Parameters_df_D'
    # there might be duplicates

ext_idx_D = []

DF1_np_D = Parameters_df_D.to_numpy()
DF2_np_D = ori_param_D_ext.to_numpy()


for ii in tqdm(range(DF2_np_D.shape[0])):
    ROW = DF2_np_D[ii,:7]  # for 7 parameters
    idx_cnt=0
    for jj in range(DF1_np_D.shape[0]):
        if np.array_equal(ROW,DF1_np_D[jj,:7])==True:
            ext_idx_D.append(jj)
            idx_cnt+=1
        else:
            continue
    if idx_cnt!=1:
        print("Check ", ii)
    else:
        continue


print(len(ext_idx_D))


In [ ]:
ext_idx_D_set = set(ext_idx_D)

ext_idx_D_final = list(ext_idx_D_set)
ext_idx_D_final.sort()  # ascending order

print(len(ext_idx_D_final))

In [ ]:
# just copy colums 'parameters_index', and 'polarity_index'
    # run again with this dataframe
df_temp_D = pd.DataFrame()
df_temp_D.loc[:, 'parameters_index'] = ext_idx_D_final
df_temp_D.loc[:, 'polarity_index'] = 8
print(df_temp_D.head(5))

In [ ]:
check_isin = np.isin(ori_idx, ext_idx_D_final)

print(check_isin.shape)
print(np.sum(check_isin))

In [ ]:
# # setting the enviroment

# # time:
# t = np.linspace(0,15,75001)  # 15 seconds, in 0.0002 (0.2ms) jumps
# correction_to_ms = int(round(0.001/(t[1]-t[0])))
# sensory_start = 5000 * correction_to_ms
# sensory_end = 10000 * correction_to_ms

# # resting state: 
# Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1))  #units: V


In [ ]:
# running (4,298) sets to check expanded Rm in dauer


Parameters_Job = df_temp_D.copy(); print(Parameters_Job.shape)

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
male_basic_conditions = []; male_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
Sgap_male_adj = []; Schem_male_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head,[1.2, 1, 0.8], "chem")
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head,[1.2, 1, 0.8], "gap")
    Schem_male_row, mult_chem_male = ecd.adjusting_strengths(Schem_male_head,[1.2, 1, 0.8], "chem")
    Sgap_male_row, mult_gap_male =  ecd.adjusting_strengths(Sgap_male_head,[1.2, 1, 0.8], "gap")
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_head,[1.2, 1, 0.8], "chem")
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_head,[1.2, 1, 0.8], "gap")

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_male_row = sf.adjust_polarity(Schem_male_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_male_adj.append(Sgap_male_row)
    Schem_male_adj.append(Schem_male_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A
    
    I_D = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df_D.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df_D.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df_D.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df_D.I_PVC[parameter_ind], 0) # units: A (for dauer)

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Males
# =============================================================================
    sex = "Males"
    V_m_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male_row, Sgap_male_row, sex)
    basic_conditions_male = sf.basic_conditions_check(V_m_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    male_basic_conditions.append(basic_conditions_male)
    if basic_conditions_male == (True):
        male_cell_act_conditions.append(sf.cells_activity_conditions_check(V_m_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        male_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df_D.Rm[parameter_ind] * Parameters_df_D.Capacitance[parameter_ind],\
                                     Parameters_df_D.Rin[parameter_ind], Parameters_df_D.Chemical_Conductance[parameter_ind],\
                                     Parameters_df_D.Gap_Conductance[parameter_ind], I, Schem_dauer_row, Sgap_dauer_row, sex)  # changed Schem/Sgap matrices
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)  # changed to '_D' for dauer
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["male_basic_conditions"] = male_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["male_cell_act_conditions"] = male_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_male_adj"] = Schem_male_adj
Parameters_Job["Sgap_male_adj"] = Sgap_male_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_male"] = ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["male_cell_act_conditions"] == True) | (Parameters_Job["male_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))
    # mutual = herm & male & dauer


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_dauer_Rm_ext_4298_260116.csv")  # change name


In [ ]:
# select mutual parameter sets (but this time it's dauer-only)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_dauer_Rm_ext_4298_260116.csv", index_col=0)  # check name
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")

some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_dauer_Rm_ext_4298_260126.csv")  # check name

herm_cell_activity_basic_mutual = parameters_pol_samp_df[((parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True) | (parameters_pol_samp_df["dauer_basic_conditions"] == "True") | (parameters_pol_samp_df["dauer_basic_conditions"] == True))].reset_index()

herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260126.csv")  # check name
# =============================================================================


In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260126.csv"  # change name
herm_cell_activity_basic_mutual_Rm_ext_D = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df_D.csv"  # for dauer
Parameters_df_D = pd.read_csv(fname2, index_col = 0)

fname3 = r"your_directory\Parameters_combinations_df.csv"
Parameters_df = pd.read_csv(fname3, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]


Different_stimulus_Job = herm_cell_activity_basic_mutual_Rm_ext_D.copy()  # check name


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row]
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons)))
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons)))
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row], (len(neurons), len(neurons)))
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons)))
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        
        sensory_input_D = sensory_input_calc(Parameters_df_D.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I_D = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input_D,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df_D.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df_D.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df_D.I_PVC[parameter_ind], 0) # units: A (for dauer)
        
# =============================================================================
# Hermaphrodites
# =============================================================================
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        herm_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Males
# =============================================================================
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        male_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df_D.Rm[parameter_ind] * Parameters_df_D.Capacitance[parameter_ind],  Parameters_df_D.Rin[parameter_ind], Parameters_df_D.Chemical_Conductance[parameter_ind], Parameters_df_D.Gap_Conductance[parameter_ind], I_D, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["male_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["male_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(male_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["male_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["male_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(male_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260126.csv")

In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")  # change name
different_stimuli_samp_pol2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260126.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg_two(sensory_voltage_changes,
                                              different_stimuli_samp_pol1,
                                              different_stimuli_samp_pol2,
                                              r"your_directory\predicted_avoidance_dauer_ori_and_Rm_ext_260212",
                                              (1.01,-0.01))  # modified function


In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")  # change name
different_stimuli_samp_pol2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260126.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg_two2(sensory_voltage_changes,
                                               different_stimuli_samp_pol1,
                                               different_stimuli_samp_pol2,
                                               r"your_directory\predicted_avoidance_dauer_ori_and_Rm_ext_260309",
                                               (1.01,-0.01))  # modified function


In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv")  # change name
different_stimuli_samp_pol2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260126.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg_two3(sensory_voltage_changes,
                                               different_stimuli_samp_pol1,
                                               different_stimuli_samp_pol2,
                                               r"your_directory\predicted_avoidance_dauer_ori_and_Rm_ext_260408",
                                               (1.03,-0.01))  # modified function


---

In [ ]:
# load file ====================================================================
fname1 = r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv"  # change name

df_temp = pd.read_csv(fname1, index_col=0)  # check variable name

Different_stimulus_Job = df_temp.copy()
# ==============================================================================

In [ ]:
import supporting_functions as sf
import os
import random

# =============================================================================
# Initializing response arrays
# =============================================================================
herm_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

male_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

dauer_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan


# =============================================================================
# save time sequence voltage (V_h, V_m, V_d)
    # row X input_strength X 7 X time series (4-dimension)
# =============================================================================
Vseq_herm = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_male = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_dauer = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))


# =============================================================================
# Simulating sensory stimuli in varying strength and finding the activations of the interneurons
# =============================================================================
for row in tqdm(range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1)):
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons)))  # 7 X 7
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons))) 
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row],  (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))
    
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    ii = 0  # for indexing
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end),\
                             neurons.index("ASH"), sensory_input, 0, range(0,len(t)),\
                             neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        # Hermaphrodite
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
        herm_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_herm[row,ii,:,:] = V_h
        
        # Male
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
        male_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_male[row,ii,:,:] = V_m
        
        # Dauer
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
        dauer_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_dauer[row,ii,:,:] = V_d
        
        ii+=1
#         print("finished row "+str(row)+", sensory input "+str(voltage))

for sens_input in tqdm(range(len(sensory_voltage_changes))):
    Different_stimulus_Job["herm_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVA[:,sens_input]
    Different_stimulus_Job["herm_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVD[:,sens_input]
    Different_stimulus_Job["herm_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVB[:,sens_input]
    Different_stimulus_Job["herm_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_PVC[:,sens_input]
    
    Different_stimulus_Job["male_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVA[:,sens_input]
    Different_stimulus_Job["male_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVD[:,sens_input]
    Different_stimulus_Job["male_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVB[:,sens_input]
    Different_stimulus_Job["male_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_PVC[:,sens_input]
    
    Different_stimulus_Job["dauer_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVA[:,sens_input]
    Different_stimulus_Job["dauer_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVD[:,sens_input]
    Different_stimulus_Job["dauer_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVB[:,sens_input]
    Different_stimulus_Job["dauer_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_PVC[:,sens_input]

print(len(Different_stimulus_Job), "_ _ _ Done")


In [ ]:
# surface area adjusted ver.


import supporting_functions as sf
import os
import random


Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk
Parameters_df_D = pd.read_csv(r"your_directory\Parameters_combinations_df_D.csv", index_col = 0)  # dauer adjusted


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

male_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

dauer_AVA = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVD = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVB = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_PVC = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan


# =============================================================================
# save time sequence voltage (V_h, V_m, V_d)
    # row X input_strength X 7 X time series (4-dimension)
# =============================================================================
Vseq_herm = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_male = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_dauer = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))


# =============================================================================
# Simulating sensory stimuli in varying strength and finding the activations of the interneurons
# =============================================================================
for row in tqdm(range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1)):
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons)))  # 7 X 7
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons))) 
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row],  (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))
    
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    ii = 0  # for indexing
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end),\
                             neurons.index("ASH"), sensory_input, 0, range(0,len(t)),\
                             neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        I_D = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df_D.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df_D.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df_D.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df_D.I_PVC[parameter_ind], 0) # units: A (for dauer)
        
        # Hermaphrodite
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
        herm_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_herm[row,ii,:,:] = V_h
        
        # Male
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
        male_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_male[row,ii,:,:] = V_m
        
        # Dauer
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df_D.Rm[parameter_ind] * Parameters_df_D.Capacitance[parameter_ind],  Parameters_df_D.Rin[parameter_ind], Parameters_df_D.Chemical_Conductance[parameter_ind], Parameters_df_D.Gap_Conductance[parameter_ind], I_D, Schem_dauer, Sgap_dauer)
        dauer_AVA[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVD[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVB[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_PVC[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_dauer[row,ii,:,:] = V_d
        
        ii+=1
#         print("finished row "+str(row)+", sensory input "+str(voltage))

for sens_input in tqdm(range(len(sensory_voltage_changes))):
    Different_stimulus_Job["herm_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVA[:,sens_input]
    Different_stimulus_Job["herm_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVD[:,sens_input]
    Different_stimulus_Job["herm_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVB[:,sens_input]
    Different_stimulus_Job["herm_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_PVC[:,sens_input]
    
    Different_stimulus_Job["male_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVA[:,sens_input]
    Different_stimulus_Job["male_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVD[:,sens_input]
    Different_stimulus_Job["male_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVB[:,sens_input]
    Different_stimulus_Job["male_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_PVC[:,sens_input]
    
    Different_stimulus_Job["dauer_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVA[:,sens_input]
    Different_stimulus_Job["dauer_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVD[:,sens_input]
    Different_stimulus_Job["dauer_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVB[:,sens_input]
    Different_stimulus_Job["dauer_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_PVC[:,sens_input]

print(len(Different_stimulus_Job), "_ _ _ Done")


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\241022_interneurons_activations_herm_male_dauer_667_260202.csv")

## Dauer with male's I-to-M connectivity

In [ ]:
# import extract_connectivity_df as ecd
# import supporting_functions as sf

# path_to_cook_file = r"your_directory\SI_5_cook_original.xlsx"

# herm_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite chemical", header = None)
# herm_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "hermaphrodite gap jn symmetric", header = None)
# male_chem_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male chemical", header = None)
# male_gapj_full_connectivity = pd.read_excel(path_to_cook_file, sheet_name = "male gap jn symmetric", header = None)

# # =============================================================================
# # Define neurons of interest:
# # In circuit_neurons the keys are the name of the neural class, and the values are
# # the regex expressions to find the cells in Cook's data.  
# # =============================================================================
# circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]',\
#                    'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
# neurons = list(circuit_neurons.keys())

# Sgap_herm_head,Schem_herm_head,Sgap_male_head,Schem_male_head = ecd.final_connectivity_matrix(herm_chem_full_connectivity,\
#                                                                                               herm_gapj_full_connectivity,\
#                                                                                               male_chem_full_connectivity,\
#                                                                                               male_gapj_full_connectivity,\
#                                                                                               circuit_neurons)


In [ ]:
# reshape dauer nociception circuit into Pechuk order
    # Original order = ASH, AVA, AVB, AVD, PVC
    #   Pechuk order = ASH, AVA, AVD, AVB, PVC
    # assume that connectivity to A and B is similar to male (i.e., use male's data) !!!

Schem_dauer_M_head = np.zeros((7,7))  # mind the direction (transpose)
Sgap_dauer_M_head = np.zeros((7,7))  # mind the direction (transpose)

# chemical
dauer_chem_adjusted = np.copy( (dauer_noci_chem_merge_norm[np.ix_(new_order,new_order)] * mult_norm[0,1]) )  # Pechuk order
Schem_dauer_M_head[:5,:5] = np.copy(dauer_chem_adjusted.T)  # transpose
Schem_dauer_M_head[5:,:] = np.copy(Schem_male_head[5:,:])  # A, B row (from male)
Schem_dauer_M_head[:,5:] = np.copy(Schem_male_head[:,5:])  # A, B column (from male)

# electrical (don't need to adjust strength; strength by section number)
dauer_gap_adjusted = np.copy( dauer_noci_gap_merge[np.ix_(new_order,new_order)] )  # Pechuk order
Sgap_dauer_M_head[:5,:5] = np.copy(dauer_gap_adjusted.T)
Sgap_dauer_M_head[5:,:] = np.copy(Sgap_male_head[5:,:])
Sgap_dauer_M_head[:,5:] = np.copy(Sgap_male_head[:,5:])


---

In [ ]:
# initialize datafrmae (118)

# =============================================================================
Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)  # original combination from Pechuk

pol_fname = r"your_directory\Polarity_combinations_sample.pkl"
possible_combinations_df = pd.read_pickle(pol_fname)

df_temp = pd.read_csv(r"your_directory\valid_667_initial_df.csv", index_col = 0); print(df_temp.shape)
print(df_temp.head(5))
# =============================================================================

In [ ]:
# running 667 set to check basic condition of dauer
    # check; Schem_dauer_M_head, Sgap_dauer_M_head (male ver.)

Parameters_Job = df_temp.copy()

# =============================================================================
# initializing the conditions
# =============================================================================
herm_basic_conditions = []; herm_cell_act_conditions = []
dauer_basic_conditions = []; dauer_cell_act_conditions = []
male_basic_conditions = []; male_cell_act_conditions = []
Sgap_herm_adj = []; Schem_herm_adj = []
Sgap_dauer_adj = []; Schem_dauer_adj = []
Sgap_male_adj = []; Schem_male_adj = []
time_constant = []

row_cnt = 0; iter_cnt = 1
CUT = 1* 10**1
iter_num = int(Parameters_Job.shape[0] / CUT) + 1

rand_row = np.array(Parameters_Job.index)
for row in rand_row:
    parameter_ind = Parameters_Job.parameters_index[row]
    polarity_ind = Parameters_Job.polarity_index[row]
# =============================================================================
# Randomly adjusting the strength of the connections
# =============================================================================
    Schem_herm_row, mult_chem_herm = ecd.adjusting_strengths(Schem_herm_head,[1.2, 1, 0.8], "chem")
    Sgap_herm_row, mult_gap_herm =  ecd.adjusting_strengths(Sgap_herm_head,[1.2, 1, 0.8], "gap")
    Schem_male_row, mult_chem_male = ecd.adjusting_strengths(Schem_male_head,[1.2, 1, 0.8], "chem")
    Sgap_male_row, mult_gap_male =  ecd.adjusting_strengths(Sgap_male_head,[1.2, 1, 0.8], "gap")
    Schem_dauer_row, mult_chem_dauer =  ecd.adjusting_strengths(Schem_dauer_M_head,[1.2, 1, 0.8], "chem")  # male ver.
    Sgap_dauer_row, mult_gap_dauer =  ecd.adjusting_strengths(Sgap_dauer_M_head,[1.2, 1, 0.8], "gap")  # male ver.

# =============================================================================
# Update polarity
# =============================================================================
    Schem_herm_row = sf.adjust_polarity(Schem_herm_row, neurons, polarity_ind, possible_combinations_df)
    Schem_male_row = sf.adjust_polarity(Schem_male_row, neurons, polarity_ind, possible_combinations_df)
    Schem_dauer_row = sf.adjust_polarity(Schem_dauer_row, neurons, polarity_ind, possible_combinations_df)

    Sgap_herm_adj.append(Sgap_herm_row)
    Schem_herm_adj.append(Schem_herm_row)
    Sgap_male_adj.append(Sgap_male_row)
    Schem_male_adj.append(Schem_male_row)
    Sgap_dauer_adj.append(Sgap_dauer_row)
    Schem_dauer_adj.append(Schem_dauer_row)

# =============================================================================
# Define input current:
# =============================================================================
    I = sf.input_current(Sgap_herm_row, t, range(sensory_start, sensory_end), neurons.index("ASH"),\
                         Parameters_df.Sensory_Input[parameter_ind], 0, range(0,len(t)), neurons.index("AVA"),\
                         Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)), neurons.index("AVB"),\
                         Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),  neurons.index("PVC"),\
                         Parameters_df.I_PVC[parameter_ind], 0) # units: A

# =============================================================================
# Hermaphrodites
# =============================================================================
    sex = "Hermaphrodites"
    V_h_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm_row, Sgap_herm_row, sex)
    basic_conditions_herm = sf.basic_conditions_check(V_h_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    herm_basic_conditions.append(basic_conditions_herm)
    if basic_conditions_herm == (True):
        herm_cell_act_conditions.append(sf.cells_activity_conditions_check(V_h_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        herm_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Males
# =============================================================================
    sex = "Males"
    V_m_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male_row, Sgap_male_row, sex)
    basic_conditions_male = sf.basic_conditions_check(V_m_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    male_basic_conditions.append(basic_conditions_male)
    if basic_conditions_male == (True):
        male_cell_act_conditions.append(sf.cells_activity_conditions_check(V_m_adj_con, neurons, sensory_start, sensory_end, correction_to_ms))
    else:
        male_cell_act_conditions.append(("irrelevant"))
# =============================================================================
# Dauers
# =============================================================================
    sex = "Hermaphrodites"  # this is correct
    V_d_adj_con = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],\
                                     Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind],\
                                     Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer_row, Sgap_dauer_row, sex)  # male ver. already applied
    basic_conditions_dauer = sf.basic_conditions_check(V_d_adj_con, t, neurons, sensory_start, sensory_end, correction_to_ms)
    dauer_basic_conditions.append(basic_conditions_dauer)
    if basic_conditions_dauer == (True):
        dauer_cell_act_conditions.append(sf.cells_activity_conditions_check(V_d_adj_con, neurons,sensory_start, sensory_end, correction_to_ms))
    else:
        dauer_cell_act_conditions.append(("irrelevant"))

# =============================================================================
# General
# =============================================================================
    time_constant.append(Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind] < 500 * 10 **-3)
    row_cnt += 1
    
    if row_cnt % CUT == 0:
        print("_ _ _ finished row", row_cnt, "_ _ _", iter_cnt, "/", iter_num)
        iter_cnt += 1
    else:
        continue

print("Done")

# =============================================================================
# Save to dataframe
# =============================================================================
Parameters_Job["herm_basic_conditions"] = herm_basic_conditions
Parameters_Job["male_basic_conditions"] = male_basic_conditions
Parameters_Job["dauer_basic_conditions"] = dauer_basic_conditions
Parameters_Job["herm_cell_act_conditions"] = herm_cell_act_conditions
Parameters_Job["male_cell_act_conditions"] = male_cell_act_conditions
Parameters_Job["dauer_cell_act_conditions"] = dauer_cell_act_conditions
Parameters_Job["time_constant"] = time_constant
Parameters_Job["Schem_herm_adj"] = Schem_herm_adj
Parameters_Job["Sgap_herm_adj"] = Sgap_herm_adj
Parameters_Job["Schem_male_adj"] = Schem_male_adj
Parameters_Job["Sgap_male_adj"] = Sgap_male_adj
Parameters_Job["Schem_dauer_adj"] = Schem_dauer_adj
Parameters_Job["Sgap_dauer_adj"] = Sgap_dauer_adj
Parameters_Job["True_herm"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True"))
Parameters_Job["True_male"] = ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["male_cell_act_conditions"] == True) | (Parameters_Job["male_cell_act_conditions"] == "True"))
Parameters_Job["True_dauer"] = ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True")) & ((Parameters_Job["dauer_cell_act_conditions"] == True) | (Parameters_Job["dauer_cell_act_conditions"] == "True"))  
Parameters_Job["mutual_basic_herm_activity"] = ((Parameters_Job["herm_basic_conditions"] == True) | (Parameters_Job["herm_basic_conditions"] == "True")) & ((Parameters_Job["herm_cell_act_conditions"] == True) | (Parameters_Job["herm_cell_act_conditions"] == "True")) & ((Parameters_Job["male_basic_conditions"] == True) | (Parameters_Job["male_basic_conditions"] == "True")) & ((Parameters_Job["dauer_basic_conditions"] == True) | (Parameters_Job["dauer_basic_conditions"] == "True"))
    # mutual = herm & male & dauer


In [ ]:
# save to csv

Parameters_Job.to_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_M_667.csv")  # change name
    # Note the 'M' at the end of file name


---

In [ ]:
# select mutual parameter sets (hermaphrodite, male, dauer_M)

from os import path
import random
import supporting_functions as sf

# =============================================================================
single_run_df = pd.read_csv(r"your_directory\parameters_basic_conditions_herm_male_dauer_M_667.csv", index_col=0)  # change name
single_run_df["parameters_index"] = single_run_df["parameters_index"].astype("int32")
single_run_df["polarity_index"] = single_run_df["polarity_index"].astype("int8")
some_true =  single_run_df[((single_run_df["herm_basic_conditions"] == "True") | (single_run_df["herm_basic_conditions"] == True) | (single_run_df["male_basic_conditions"] == "True") | (single_run_df["male_basic_conditions"] == True) | (single_run_df["dauer_basic_conditions"] == "True") | (single_run_df["dauer_basic_conditions"] == True)) & (single_run_df["time_constant"] == True)] 
    # herm, male, dauer_M

parameters_pol_samp_df = some_true.copy()
print(parameters_pol_samp_df.info())
parameters_pol_samp_df.to_csv(r"your_directory\basic_true_par_pol_samp_herm_male_dauer_M_667.csv")
herm_cell_activity_basic_mutual = parameters_pol_samp_df[(parameters_pol_samp_df["mutual_basic_herm_activity"] == "True") | (parameters_pol_samp_df["mutual_basic_herm_activity"] == True)].reset_index()
herm_cell_activity_basic_mutual.to_csv(r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_M_667.csv")
# =============================================================================
    # Note the 'M' at the end of file names


In [ ]:
# # =============================================================================
# # Define for cluster use
# # =============================================================================
# job_ID = int(os.environ['LSB_JOBINDEX']) - 1
job_num = 2000

# # =============================================================================
# # Read parameters sets
# # this file contains all sets that were appropriate to either of the sexes, 
# # after the adjustments to the connections' strength. 
# # =============================================================================
fname1 = r"your_directory\herm_cell_activity_basic_mutual_herm_male_dauer_M_667.csv"  # filtered mutual parameter sets
herm_cell_activity_basic_mutual_M_667 = pd.read_csv(fname1)  # check variable name

fname2 = r"your_directory\Parameters_combinations_df.csv"  # original
Parameters_df = pd.read_csv(fname2, index_col = 0)


In [ ]:
# =============================================================================
# Set environment
# =============================================================================
# define neurons of interest:
circuit_neurons = {'ASH':'ASH[RL]', 'AVA':'AVA[RL]', 'AVD':'AVD[RL]', 'AVB':'AVB[RL]', 'PVC':'PVC[RL]', 'A':'[DV]A[0-9]', 'B':'[DV]B[0-9]'}
neurons = list(circuit_neurons.keys())

# time:
t = np.linspace(0,15,75001) # 15 seconds, in 0.0002 (0.2ms) jumps
correction_to_ms = int(round(0.001/(t[1]-t[0])))
sensory_start = 5000 * correction_to_ms
sensory_end = 10000 * correction_to_ms

# resting state: 
Vrest = np.reshape(np.repeat(np.array([-0.04]),len(neurons)), (len(neurons),1)) #units: V

# =============================================================================
# Input that would trigger a specific voltage change:
# =============================================================================
# V = IR, I = V/R
sensory_input_calc = lambda res, volt: volt/res
sensory_voltage_changes = [75 * 10 **-3, 65 * 10 **-3, 55 * 10 ** -3, 45 * 10 **-3, 35 * 10 **-3, 25 * 10 ** -3, 15 * 10 **-3, 5 * 10 ** -3, 1 * 10 **-3]

Different_stimulus_Job = herm_cell_activity_basic_mutual_M_667.copy()  # check name


# =============================================================================
# Initializing response arrays
# =============================================================================
herm_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_response = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

# =============================================================================
# Simulating sensory stimuli in varying strength
# =============================================================================
row_cnt = 0
for row in range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1):
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons))) 
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons)))
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row], (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))    
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end), neurons.index("ASH"), sensory_input,\
                             0, range(0,len(t)), neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind],\
                             0, range(0,len(t)), neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind],\
                             0, range(0,len(t)), neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        
# =============================================================================
# Hermaphrodites
# =============================================================================
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        herm_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Males
# =============================================================================
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        male_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])

# =============================================================================
# Dauers
# =============================================================================
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
# since we expect oscillations we check the average of 1 second to determine which is higher, and not a single time step. 
        dauer_response[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("A"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("B"), sensory_end - (1000 * correction_to_ms) : sensory_end])
    
    row_cnt += 1
    
    if row_cnt % 10**1 == 0:
        print("finished row "+str(row_cnt))
    else:
        continue

print(row_cnt, "Done")


for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["herm_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["male_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
    Different_stimulus_Job["dauer_reponse_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_response[:,sens_input]

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["herm_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(herm_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["male_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(male_response[:,sens_input]> 0, 1, 0)

for sens_input in range(len(sensory_voltage_changes)):
     Different_stimulus_Job["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = np.where(dauer_response[:,sens_input]> 0, 1, 0)


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_M_667.csv")

In [ ]:
# plot

from os import path
import seaborn as sns
import supporting_functions as sf


different_stimuli_samp_pol_M = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_M_667.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_all(sensory_voltage_changes,\
                                       different_stimuli_samp_pol_M,\
                                       r"your_directory\predicted_avoidance_adult_male_dauer_M_667",\
                                       (1.02,-0.18))  # modified function


In [ ]:
# (modified from supporting function 'plot_reverse_probability_per_input') ====================================================================

    # manuscript ver.
    # svg ; change color ; change font
    # for Male connectivity dauer


def plot_reverse_probability_per_input_AD_svg_M(sensory_voltage_changes, responses_df_all1, responses_df_all2, title, anchor, *args):
    '''
    Inputs:
        1. sensory_voltage_changes - list of voltage changes to the sensory neuron that were tested.
        2. responses_df_all - df with the WT responses of hermaphrodite, male, and dauer
        3. title
        4. anchor - for legend
    args:
    optional. for connectivity perturbations, it is possible to add additional curves with the 
    responses of the masculinized\feminized worms. 
        1. dimorphic_df_change (args[i * 4])
        2. dimorphic_change (args[i * 4 + 1])
        3. label (args[i * 4 + 2])
        4. additional_color (args[i * 4 + 3])
        5. if args > 4, repeat 1-4. 
    
    Output:
       plot (saved and presented) 
    '''
    
    fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))
    
    mean_reversal_herm = []
    mean_reversal_dauer = []
    mean_reversal_dauer_M = []
    for sens_voltage in range(len(sensory_voltage_changes)):
        mean_reversal_herm.append(np.mean(responses_df_all1["herm_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer.append(np.mean(responses_df_all1["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
        mean_reversal_dauer_M.append(np.mean(responses_df_all2["dauer_A_vs_B_"+str(sensory_voltage_changes[sens_voltage])+"mV_change"]))
    plot_margins = 2
    if len(args)>0:
        dimorphic_df_change = []
        dimorphic_changes = []
        labels = []
        additional_colors =[]
        for i in range(int(len(args)/4)):
            dimorphic_df_change.append(args[i * 4])
            dimorphic_changes.append(args[i * 4 + 1])
            labels.append(args[i * 4 + 2])
            additional_colors.append(args[i * 4 + 3])
            mean_reversal_changed_con = {}
        for change in range(len(dimorphic_changes)):
            mean_reversal_changed_con[labels[change]] = []
            for sens_voltage in range(len(sensory_voltage_changes)):
                mean_reversal_changed_con[labels[change]] += [np.mean(dimorphic_df_change[change]["A_vs_B_"+ dimorphic_changes[change] + "_" +str(sensory_voltage_changes[sens_voltage])+"mV_change"])]
            plt.plot(sensory_voltage_changes,mean_reversal_changed_con[labels[change]], marker = "o", label = labels[change], color = additional_colors[change])
    
    plt.plot(sensory_voltage_changes,mean_reversal_herm, marker = "o", label = "Hermaphrodites", color = '#1F77B4')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer, marker = "o", label = "Dauers", color ='#FF7F0E')
    plt.plot(sensory_voltage_changes,mean_reversal_dauer_M, marker = "o", label = "Dauers", color ='#D62728')
    plt.xlim([min(sensory_voltage_changes) - plot_margins, max(sensory_voltage_changes) + plot_margins])
    plt.ylabel("Predicted Avoidance", fontsize = 14, fontname='sans-serif')
    plt.xlabel(r'$\Delta$V in sensory neuron (mV)', fontsize=14, fontname='sans-serif')
    if len(args)>0:
        plt.legend(labels + ["Wild-type adults", "Wild-type dauers", "Dauers (male connectivity)"] , fontsize = 14, loc='lower left', bbox_to_anchor = anchor)
    else:
        plt.legend(["Wild-type adults", "Wild-type dauers", "Dauers (male connectivity)"] , fontsize = 14, loc='lower left', bbox_to_anchor = anchor)
    
    if "\\" in title:
        title_on_plot = title.split("\\")[-1]
    else: title_on_plot == title
    
#     plt.title(title_on_plot, fontsize = 18, fontname='sans-serif')
    plt.savefig(title +".svg", dpi=300, facecolor='w', bbox_inches='tight')
    plt.show()

# =============================================================================================================================================

In [ ]:
# (plot)

different_stimuli_temp1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667.csv")  # change name
different_stimuli_temp2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_M_667.csv")  # change name

# Here we add 1 to the polarity index such that the first index would be 1 instead of 0. this is the only place in which it happens. 
sensory_voltage_changes = [1.0, 5.0, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0]

# plot all
plot_reverse_probability_per_input_AD_svg_M(sensory_voltage_changes,
                                            different_stimuli_temp1, different_stimuli_temp2,
                                            r"your_directory\Dauer with Male I-to-M connectivity",
                                            (1.0,0.0))  # modified function


---

In [ ]:
# load file ====================================================================
fname1 = r"your_directory\different_stimuli_samp_pol_herm_male_dauer_M_667.csv"  # change name
df_temp = pd.read_csv(fname1, index_col=0)  # check variable name

Different_stimulus_Job = df_temp.copy()
# ==============================================================================

In [ ]:
import supporting_functions as sf
import os
import random

# =============================================================================
# Initializing response arrays
# =============================================================================
herm_AVA_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVD_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_AVB_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
herm_PVC_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

male_AVA_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVD_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_AVB_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
male_PVC_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan

dauer_AVA_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVD_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_AVB_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan
dauer_PVC_M = np.empty((len(Different_stimulus_Job),len(sensory_voltage_changes))) * np.nan


# =============================================================================
# save time sequence voltage (V_h, V_m, V_d)
    # row X input_strength X 7 X time series (4-dimension)
# =============================================================================
Vseq_herm_M = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_male_M = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))
Vseq_dauer_M = np.zeros((len(Different_stimulus_Job),len(sensory_voltage_changes),7,t.size))


# =============================================================================
# Simulating sensory stimuli in varying strength and finding the activations of the interneurons
# =============================================================================
for row in tqdm(range(Different_stimulus_Job.index[0],Different_stimulus_Job.index[len(Different_stimulus_Job)-1]+1)):
    Sgap_herm = sf.str_to_nparray(Different_stimulus_Job.Sgap_herm_adj[row], (len(neurons), len(neurons)))  # 7 X 7
    Schem_herm = sf.str_to_nparray(Different_stimulus_Job.Schem_herm_adj[row],  (len(neurons), len(neurons))) 
    Sgap_male = sf.str_to_nparray(Different_stimulus_Job.Sgap_male_adj[row],  (len(neurons), len(neurons))) 
    Schem_male = sf.str_to_nparray(Different_stimulus_Job.Schem_male_adj[row],  (len(neurons), len(neurons)))
    Sgap_dauer = sf.str_to_nparray(Different_stimulus_Job.Sgap_dauer_adj[row],  (len(neurons), len(neurons))) 
    Schem_dauer = sf.str_to_nparray(Different_stimulus_Job.Schem_dauer_adj[row],  (len(neurons), len(neurons)))
    
    parameter_ind = Different_stimulus_Job.parameters_index[row] 
    ii = 0  # for indexing
    for voltage in range(len(sensory_voltage_changes)):
        sensory_input = sensory_input_calc(Parameters_df.Rin[parameter_ind], sensory_voltage_changes[voltage])
        I = sf.input_current(Sgap_herm, t, range(sensory_start, sensory_end),\
                             neurons.index("ASH"), sensory_input, 0, range(0,len(t)),\
                             neurons.index("AVA"), Parameters_df.I_AVA[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("AVB"), Parameters_df.I_AVB[parameter_ind], 0, range(0,len(t)),\
                             neurons.index("PVC"), Parameters_df.I_PVC[parameter_ind], 0) # units: A
        # Hermaphrodite
        V_h = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_herm, Sgap_herm)
        herm_AVA_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVD_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_AVB_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        herm_PVC_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_h[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_h[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_herm_M[row,ii,:,:] = V_h
        
        # Male
        V_m = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_male, Sgap_male)
        male_AVA_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVD_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_AVB_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        male_PVC_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_m[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_m[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_male_M[row,ii,:,:] = V_m
        
        # Dauer
        V_d = sf.neurons_elegans(t, Vrest, Parameters_df.Rm[parameter_ind] * Parameters_df.Capacitance[parameter_ind],  Parameters_df.Rin[parameter_ind], Parameters_df.Chemical_Conductance[parameter_ind], Parameters_df.Gap_Conductance[parameter_ind], I, Schem_dauer, Sgap_dauer)
        dauer_AVA_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVA"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVA"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVD_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVD"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVD"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_AVB_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("AVB"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("AVB"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        dauer_PVC_M[row - Different_stimulus_Job.index[0], voltage] = np.mean(V_d[neurons.index("PVC"), sensory_end - (1000 * correction_to_ms) : sensory_end]) - np.mean(V_d[neurons.index("PVC"), sensory_start - (1000 * correction_to_ms) : sensory_start])
        Vseq_dauer_M[row,ii,:,:] = V_d
        
        ii+=1
#         print("finished row "+str(row)+", sensory input "+str(voltage))

for sens_input in tqdm(range(len(sensory_voltage_changes))):
    Different_stimulus_Job["herm_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVA_M[:,sens_input]
    Different_stimulus_Job["herm_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVD_M[:,sens_input]
    Different_stimulus_Job["herm_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_AVB_M[:,sens_input]
    Different_stimulus_Job["herm_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = herm_PVC_M[:,sens_input]
    
    Different_stimulus_Job["male_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVA_M[:,sens_input]
    Different_stimulus_Job["male_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVD_M[:,sens_input]
    Different_stimulus_Job["male_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_AVB_M[:,sens_input]
    Different_stimulus_Job["male_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = male_PVC_M[:,sens_input]
    
    Different_stimulus_Job["dauer_AVA_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVA_M[:,sens_input]
    Different_stimulus_Job["dauer_AVD_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVD_M[:,sens_input]
    Different_stimulus_Job["dauer_AVB_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_AVB_M[:,sens_input]
    Different_stimulus_Job["dauer_PVC_" + str(sensory_voltage_changes[sens_input]* 10**3) +"mV_change"] = dauer_PVC_M[:,sens_input]

print(len(Different_stimulus_Job), "_ _ _ Done")


In [ ]:
Different_stimulus_Job.to_csv(r"your_directory\241023_interneurons_activations_herm_male_dauer_M_667.csv")

---

## Parameter distribution

In [ ]:
# load data

# original combination from Pechuk et al. =====================================================================================================
Parameters_df = pd.read_csv(r"your_directory\Parameters_combinations_df.csv", index_col = 0)
pol_fname = r"your_directory\Polarity_combinations_sample.pkl"
possible_combinations_df = pd.read_pickle(pol_fname)
# =============================================================================================================================================


# indices
herm_male_667_df = pd.read_csv(r"your_directory\valid_667_initial_df.csv", index_col=0)
print(herm_male_667_df.head(5), "\n", herm_male_667_df.shape, "\n\n")
herm_dauer_187_df = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_250716.csv", index_col=0)
print(herm_dauer_187_df.shape)
herm_dauer_193_df = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv", index_col=0)
print(herm_dauer_193_df.shape)


In [ ]:
Parameters_df.shape # index starts with '0'

In [ ]:
list(Parameters_df.columns)

In [ ]:
# herm + male parameters (667)

HM_idx = list(herm_male_667_df['parameters_index'])  # list of indices

HM_param_list = Parameters_df.iloc[HM_idx]
print(HM_param_list.shape)


In [ ]:
HM_param_list.head(5)

In [ ]:
# herm + dauer parameters (187)

HD_idx = list(herm_dauer_187_df['parameters_index'])  # list of indices

HD_param_list = Parameters_df.iloc[HD_idx]
print(HD_param_list.shape)


In [ ]:
# herm + dauer parameters (193)

HD_idx2 = list(herm_dauer_193_df['parameters_index'])  # list of indices

HD_param_list2 = Parameters_df.iloc[HD_idx2]
print(HD_param_list2.shape)


In [ ]:
HD_param_list.head(5)

In [ ]:
HD_param_list2.head(5)

In [ ]:
DIFF_idx = list(set(HM_idx)-set(HD_idx))
print(len(HM_idx), " ", len(HD_idx), " ", len(DIFF_idx))

In [ ]:
DIFF_idx2 = list(set(HM_idx)-set(HD_idx2))
print(len(HM_idx), " ", len(HD_idx2), " ", len(DIFF_idx2))

In [ ]:
DIFF_param_list = Parameters_df.iloc[DIFF_idx]
print(DIFF_param_list.shape)

In [ ]:
DIFF_param_list2 = Parameters_df.iloc[DIFF_idx2]
print(DIFF_param_list2.shape)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
# compare Rm (membrane resistance)

HM_Rm = np.array(HM_param_list['Rm'])
HD_Rm = np.array(HD_param_list2['Rm'])
DIFF_Rm = np.array(DIFF_param_list2['Rm'])

Rm_x = list(Parameters_df['Rm'].unique())[::-1]  # small -> large val.

HM_Rm_cnt = []
HD_Rm_cnt = []
DIFF_Rm_cnt = []
for vals in Rm_x:
    HM_Rm_cnt.append(np.where(HM_Rm == vals)[0].size)
    HD_Rm_cnt.append(np.where(HD_Rm == vals)[0].size)
    DIFF_Rm_cnt.append(np.where(DIFF_Rm == vals)[0].size)


In [ ]:
# compare Cm (membrane capacitance)

HM_Cm = np.array(HM_param_list['Capacitance'])
HD_Cm = np.array(HD_param_list2['Capacitance'])
DIFF_Cm = np.array(DIFF_param_list2['Capacitance'])

Cm_x = list(Parameters_df['Capacitance'].unique())  # small -> large val.

HM_Cm_cnt = []
HD_Cm_cnt = []
DIFF_Cm_cnt = []
for vals in Cm_x:
    HM_Cm_cnt.append(np.where(HM_Cm == vals)[0].size)
    HD_Cm_cnt.append(np.where(HD_Cm == vals)[0].size)
    DIFF_Cm_cnt.append(np.where(DIFF_Cm == vals)[0].size)


In [ ]:
# compare Chemical conductance

HM_C_con = np.array(HM_param_list['Chemical_Conductance'])
HD_C_con = np.array(HD_param_list2['Chemical_Conductance'])
DIFF_C_con = np.array(DIFF_param_list2['Chemical_Conductance'])

C_con_x = list(Parameters_df['Chemical_Conductance'].unique())[::-1]  # small -> large val.

HM_C_con_cnt = []
HD_C_con_cnt = []
DIFF_C_con_cnt = []
for vals in C_con_x:
    HM_C_con_cnt.append(np.where(HM_C_con == vals)[0].size)
    HD_C_con_cnt.append(np.where(HD_C_con == vals)[0].size)
    DIFF_C_con_cnt.append(np.where(DIFF_C_con == vals)[0].size)


In [ ]:
# compare Gap conductance

HM_G_con = np.array(HM_param_list['Gap_Conductance'])
HD_G_con = np.array(HD_param_list2['Gap_Conductance'])
DIFF_G_con = np.array(DIFF_param_list2['Gap_Conductance'])

G_con_x = list(Parameters_df['Gap_Conductance'].unique())[::-1]  # small -> large val.

HM_G_con_cnt = []
HD_G_con_cnt = []
DIFF_G_con_cnt = []
for vals in G_con_x:
    HM_G_con_cnt.append(np.where(HM_G_con == vals)[0].size)
    HD_G_con_cnt.append(np.where(HD_G_con == vals)[0].size)
    DIFF_G_con_cnt.append(np.where(DIFF_G_con == vals)[0].size)


In [ ]:
# compare Basal input AVA

HM_dvAVA = np.array(HM_param_list['Voltage_Change_AVA'])
HD_dvAVA = np.array(HD_param_list2['Voltage_Change_AVA'])
DIFF_dvAVA = np.array(DIFF_param_list2['Voltage_Change_AVA'])

dvAVA_x = list(Parameters_df['Voltage_Change_AVA'].unique())[::-1]  # small -> large val.

HM_dvAVA_cnt = []
HD_dvAVA_cnt = []
DIFF_dvAVA_cnt = []
for vals in dvAVA_x:
    HM_dvAVA_cnt.append(np.where(HM_dvAVA == vals)[0].size)
    HD_dvAVA_cnt.append(np.where(HD_dvAVA == vals)[0].size)
    DIFF_dvAVA_cnt.append(np.where(DIFF_dvAVA == vals)[0].size)


In [ ]:
# compare Basal input AVB

HM_dvAVB = np.array(HM_param_list['Voltage_Change_AVB'])
HD_dvAVB = np.array(HD_param_list2['Voltage_Change_AVB'])
DIFF_dvAVB = np.array(DIFF_param_list2['Voltage_Change_AVB'])

dvAVB_x = list(Parameters_df['Voltage_Change_AVB'].unique())[::-1]  # small -> large val.

HM_dvAVB_cnt = []
HD_dvAVB_cnt = []
DIFF_dvAVB_cnt = []
for vals in dvAVB_x:
    HM_dvAVB_cnt.append(np.where(HM_dvAVB == vals)[0].size)
    HD_dvAVB_cnt.append(np.where(HD_dvAVB == vals)[0].size)
    DIFF_dvAVB_cnt.append(np.where(DIFF_dvAVB == vals)[0].size)


In [ ]:
# compare Basal input PVC

HM_dvPVC = np.array(HM_param_list['Voltage_Change_PVC'])
HD_dvPVC = np.array(HD_param_list2['Voltage_Change_PVC'])
DIFF_dvPVC = np.array(DIFF_param_list2['Voltage_Change_PVC'])

dvPVC_x = list(Parameters_df['Voltage_Change_PVC'].unique())[::-1]  # small -> large val.

HM_dvPVC_cnt = []
HD_dvPVC_cnt = []
DIFF_dvPVC_cnt = []
for vals in dvPVC_x:
    HM_dvPVC_cnt.append(np.where(HM_dvPVC == vals)[0].size)
    HD_dvPVC_cnt.append(np.where(HD_dvPVC == vals)[0].size)
    DIFF_dvPVC_cnt.append(np.where(DIFF_dvPVC == vals)[0].size)


In [ ]:
# bar graph


# mean
HD_Rm_mean = np.mean(HD_Rm); DIFF_Rm_mean = np.mean(DIFF_Rm)
HD_Cm_mean = np.mean(HD_Cm); DIFF_Cm_mean = np.mean(DIFF_Cm)
HD_G_con_mean = np.mean(HD_G_con); DIFF_G_con_mean = np.mean(DIFF_G_con)
HD_C_con_mean = np.mean(HD_C_con); DIFF_C_con_mean = np.mean(DIFF_C_con)

# error (std)
HD_Rm_std = np.std(HD_Rm); DIFF_Rm_std = np.std(DIFF_Rm)
HD_Cm_std = np.std(HD_Cm); DIFF_Cm_std = np.std(DIFF_Cm)
HD_G_con_std = np.std(HD_G_con); DIFF_G_con_std = np.std(DIFF_G_con)
HD_C_con_std = np.std(HD_C_con); DIFF_C_con_std = np.std(DIFF_C_con)

# error (sem)
HD_Rm_sem = stats.sem(HD_Rm); DIFF_Rm_sem = stats.sem(DIFF_Rm)
HD_Cm_sem = stats.sem(HD_Cm); DIFF_Cm_sem = stats.sem(DIFF_Cm)
HD_G_con_sem = stats.sem(HD_G_con); DIFF_G_con_sem = stats.sem(DIFF_G_con)
HD_C_con_sem = stats.sem(HD_C_con); DIFF_C_con_sem = stats.sem(DIFF_C_con)


In [ ]:
# bar graph


b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


# Rm
fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('kΩ·cm^2', fontsize=14, fontname='sans-serif')
ax1.set_title('Rm', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], DIFF_Rm_mean, yerr=DIFF_Rm_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], HD_Rm_mean, yerr=HD_Rm_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,65000)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["474","193"], fontsize=fsz, rotation=0, fontname='sans-serif')

plt.show()


# Cm
fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('µF/cm^2', fontsize=14, fontname='sans-serif')
ax1.set_title('Cm', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], DIFF_Cm_mean, yerr=DIFF_Cm_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], HD_Cm_mean, yerr=HD_Cm_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["474","193"], fontsize=fsz, rotation=0, fontname='sans-serif')

plt.show()


# C_con
fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('nS', fontsize=14, fontname='sans-serif')
ax1.set_title('G chem.', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], DIFF_C_con_mean, yerr=DIFF_C_con_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], HD_C_con_mean, yerr=HD_C_con_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,5.3*10**(-12))
ax1.set_xticks(xpos)
ax1.set_xticklabels(["474","193"], fontsize=fsz, rotation=0, fontname='sans-serif')

plt.show()


# G_con
fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('nS', fontsize=14, fontname='sans-serif')
ax1.set_title('G gap.', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], DIFF_G_con_mean, yerr=DIFF_G_con_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], HD_G_con_mean, yerr=HD_G_con_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,3.45*10**(-11))
ax1.set_xticks(xpos)
ax1.set_xticklabels(["474","193"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [3.1*10**(-11),3.1*10**(-11)], lw=0.8, c='k')
ax1.text(1.5, 3.1*10**(-11), "*", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# statistical test

from scipy.stats import mannwhitneyu
from scipy.stats import ranksums

print("Rm :", ranksums(DIFF_Rm, HD_Rm)[1])
print("C_con :", ranksums(DIFF_C_con, HD_C_con)[1])
print("Cm :", ranksums(DIFF_Cm, HD_Cm)[1])
print("dvAVA :", ranksums(DIFF_dvAVA, HD_dvAVA)[1])
print("G_con :", ranksums(DIFF_G_con, HD_G_con)[1])
print("dvAVB :", ranksums(DIFF_dvAVB, HD_dvAVB)[1])
print("dvPVC :", ranksums(DIFF_dvPVC, HD_dvPVC)[1])


### Rm_ext ver.

In [ ]:
# load data

dauer_Rm_ext_df = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260109.csv", index_col=0)
print(dauer_Rm_ext_df.shape)


In [ ]:
# Rm_ext parameters (1527)

Rm_ext_idx = list(dauer_Rm_ext_df['parameters_index'])  # list of indices

Rm_ext_param_list = Parameters_df.iloc[Rm_ext_idx]
print(Rm_ext_param_list.shape)


In [ ]:
# Rm_ext parameters (1527)

Rm_ext_param_list = Parameters_df.iloc[Rm_ext_idx]
print(Rm_ext_param_list.shape)


In [ ]:
HD_param_list.head(5)

In [ ]:
Rm_ext_param_list.head(5)

In [ ]:
Rm_ext_idx1 = list(np.setdiff1d(HM_idx,Rm_ext_idx))  # 667 - Rm_ext
Rm_ext_idx2 = list(np.setdiff1d(Rm_ext_idx,HM_idx))  # Rm_ext - 667
    # overlap = 488

print(len(Rm_ext_idx1), " ", len(Rm_ext_idx2), " ", len(Rm_ext_idx))

In [ ]:
# compare Rm (membrane resistance)

HM_Rm = np.array(HM_param_list['Rm'])
Rm_ext_Rm = np.array(Rm_ext_param_list['Rm'])

Rm_x = list(Parameters_df['Rm'].unique())[::-1]  # small -> large val.

HM_Rm_cnt = []
Rm_ext_Rm_cnt = []
for vals in Rm_x:
    HM_Rm_cnt.append(np.where(HM_Rm == vals)[0].size)
    Rm_ext_Rm_cnt.append(np.where(Rm_ext_Rm == vals)[0].size)


In [ ]:
# compare Cm (membrane capacitance)

HM_Cm = np.array(HM_param_list['Capacitance'])
Rm_ext_Cm = np.array(Rm_ext_param_list['Capacitance'])

Cm_x = list(Parameters_df['Capacitance'].unique())  # small -> large val.

HM_Cm_cnt = []
Rm_ext_Cm_cnt = []
for vals in Cm_x:
    HM_Cm_cnt.append(np.where(HM_Cm == vals)[0].size)
    Rm_ext_Cm_cnt.append(np.where(Rm_ext_Cm == vals)[0].size)


### ( surf_adj ver. )

In [ ]:
# load data

dauer_surf_adj_df1 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_herm_male_dauer_667_260116.csv", index_col=0)
print(dauer_surf_adj_df1.shape)  # surf_adj
dauer_surf_adj_df2 = pd.read_csv(r"your_directory\different_stimuli_samp_pol_dauer_Rm_ext_4298_260116.csv", index_col=0)
print(dauer_surf_adj_df2.shape)  # surf_adj + Rm_ext


In [ ]:
dauer_surf_adj_df2 = pd.read_csv(r"your_directory\herm_cell_activity_basic_mutual_dauer_Rm_ext_4298_260126.csv", index_col=0)
print(dauer_surf_adj_df2.shape)

In [ ]:
# surf_adj parameters

surf_adj1_idx = list(dauer_surf_adj_df1['parameters_index'])  # list of indices

surf_adj1_param_list = Parameters_df.iloc[surf_adj1_idx]
print(surf_adj1_param_list.shape)


surf_adj2_idx = list(dauer_surf_adj_df2['parameters_index'])  # list of indices

surf_adj2_param_list = Parameters_df.iloc[surf_adj2_idx]
print(surf_adj2_param_list.shape)


In [ ]:
surf_adj1_param_list.head(5)

In [ ]:
surf_adj2_param_list.head(5)

In [ ]:
# idx1

surf_adj1_idx1 = list(np.setdiff1d(HM_idx,surf_adj1_idx))  # 667 - surf_adj1
surf_adj1_idx2 = list(np.setdiff1d(surf_adj1_idx,HM_idx))  # surf_adj1 - 667
    # surf_adj1 : surface adjustment only

print(len(surf_adj1_idx1), " ", len(surf_adj1_idx2), " ", len(surf_adj1_idx))


In [ ]:
# idx2

surf_adj2_idx1 = list(np.setdiff1d(HM_idx,surf_adj2_idx))  # 667 - surf_adj2
surf_adj2_idx2 = list(np.setdiff1d(surf_adj2_idx,HM_idx))  # surf_adj2 - 667
    # surf_adj2 : surface adjustment + Rm ext.

print(len(surf_adj2_idx1), " ", len(surf_adj2_idx2), " ", len(surf_adj2_idx))


In [ ]:
# compare Rm (membrane resistance) - idx1

HM_Rm = np.array(HM_param_list['Rm'])
surf_adj1_Rm = np.array(surf_adj1_param_list['Rm'])

Rm_x = list(Parameters_df['Rm'].unique())[::-1]  # small -> large val.

HM_Rm_cnt = []
surf_adj1_Rm_cnt = []
for vals in Rm_x:
    HM_Rm_cnt.append(np.where(HM_Rm == vals)[0].size)
    surf_adj1_Rm_cnt.append(np.where(surf_adj1_Rm == vals)[0].size)


In [ ]:
# compare Rm (membrane resistance) - idx2

HM_Rm = np.array(HM_param_list['Rm'])
surf_adj2_Rm = np.array(surf_adj2_param_list['Rm'])

Rm_x = list(Parameters_df['Rm'].unique())[::-1]  # small -> large val.

HM_Rm_cnt = []
surf_adj2_Rm_cnt = []
for vals in Rm_x:
    HM_Rm_cnt.append(np.where(HM_Rm == vals)[0].size)
    surf_adj2_Rm_cnt.append(np.where(surf_adj2_Rm == vals)[0].size)


In [ ]:
# statistical test

from scipy.stats import mannwhitneyu
from scipy.stats import ranksums

HM_Rm_mean = np.mean(HM_Rm); surf_adj2_Rm_mean = np.mean(surf_adj2_Rm)
HM_Rm_sem = stats.sem(HM_Rm); surf_adj2_Rm_sem = stats.sem(surf_adj2_Rm)


print("Rm :", ranksums(HM_Rm, surf_adj2_Rm)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('kΩ·cm^2', fontsize=14, fontname='sans-serif')
ax1.set_title('Rm', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], HM_Rm_mean, yerr=HM_Rm_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_Rm_mean, yerr=surf_adj2_Rm_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,86000)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [76500,76500], lw=0.8, c='k')
ax1.text(1.5, 77000, "*", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare Cm - idx2

HM_Cm = np.array(HM_param_list['Capacitance'])
surf_adj2_Cm = np.array(surf_adj2_param_list['Capacitance'])


In [ ]:
# statistical test

HM_Cm_mean = np.mean(HM_Cm); surf_adj2_Cm_mean = np.mean(surf_adj2_Cm)
HM_Cm_sem = stats.sem(HM_Cm); surf_adj2_Cm_sem = stats.sem(surf_adj2_Cm)


print("Cm :", ranksums(HM_Cm, surf_adj2_Cm)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('uF/cm^2', fontsize=14, fontname='sans-serif')
ax1.set_title('Cm', fontsize=18, fontname='sans-serif')
ax1.bar(xpos[0], HM_Cm_mean, yerr=HM_Cm_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_Cm_mean, yerr=surf_adj2_Cm_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,2.8*10**(-6))
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [2.5*10**(-6),2.5*10**(-6)], lw=0.8, c='k')
ax1.text(1.5, 2.5*10**(-6), "*", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare G_con - idx2

HM_G_con = np.array(HM_param_list['Gap_Conductance'])
surf_adj2_G_con = np.array(surf_adj2_param_list['Gap_Conductance'])


In [ ]:
# statistical test

HM_G_con_mean = np.mean(HM_G_con); surf_adj2_G_con_mean = np.mean(surf_adj2_G_con)
HM_G_con_sem = stats.sem(HM_G_con); surf_adj2_G_con_sem = stats.sem(surf_adj2_G_con)


print("G_con :", ranksums(HM_G_con, surf_adj2_G_con)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('nS', fontsize=14, fontname='sans-serif')
ax1.set_title('Gap conductance', fontsize=18, fontname='sans-serif', y=1.05)
ax1.bar(xpos[0], HM_G_con_mean, yerr=HM_G_con_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_G_con_mean, yerr=surf_adj2_G_con_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,3.2*10**(-11))
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [2.85*10**(-11),2.85*10**(-11)], lw=0.8, c='k')
ax1.text(1.5, 2.85*10**(-11), "***", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare C_con - idx2

HM_C_con = np.array(HM_param_list['Chemical_Conductance'])
surf_adj2_C_con = np.array(surf_adj2_param_list['Chemical_Conductance'])


In [ ]:
# statistical test

HM_C_con_mean = np.mean(HM_C_con); surf_adj2_C_con_mean = np.mean(surf_adj2_C_con)
HM_C_con_sem = stats.sem(HM_C_con); surf_adj2_C_con_sem = stats.sem(surf_adj2_C_con)


print("C_con :", ranksums(HM_C_con, surf_adj2_C_con)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('nS', fontsize=14, fontname='sans-serif')
ax1.set_title('Chemical conductance', fontsize=18, fontname='sans-serif', y=1.05)
ax1.bar(xpos[0], HM_C_con_mean, yerr=HM_C_con_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_C_con_mean, yerr=surf_adj2_C_con_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,4.9*10**(-12))
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [4.45*10**(-12),4.45*10**(-12)], lw=0.8, c='k')
ax1.text(1.5, 4.45*10**(-12), "***", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare dvAVA - idx2

HM_dvAVA = np.array(HM_param_list['Voltage_Change_AVA'])
surf_adj2_dvAVA = np.array(surf_adj2_param_list['Voltage_Change_AVA'])


In [ ]:
# statistical test

HM_dvAVA_mean = np.mean(HM_dvAVA); surf_adj2_dvAVA_mean = np.mean(surf_adj2_dvAVA)
HM_dvAVA_sem = stats.sem(HM_dvAVA); surf_adj2_dvAVA_sem = stats.sem(surf_adj2_dvAVA)


print("dvAVA :", ranksums(HM_dvAVA, surf_adj2_dvAVA)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('mV', fontsize=14, fontname='sans-serif')
ax1.set_title('Basal input AVA', fontsize=18, fontname='sans-serif', y=1.05)
ax1.bar(xpos[0], HM_dvAVA_mean, yerr=HM_dvAVA_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_dvAVA_mean, yerr=surf_adj2_dvAVA_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,0.0029)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [0.0026,0.0026], lw=0.8, c='k')
ax1.text(1.5, 0.0026, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare dvAVB - idx2

HM_dvAVB = np.array(HM_param_list['Voltage_Change_AVB'])
surf_adj2_dvAVB = np.array(surf_adj2_param_list['Voltage_Change_AVB'])


In [ ]:
# statistical test

HM_dvAVB_mean = np.mean(HM_dvAVB); surf_adj2_dvAVB_mean = np.mean(surf_adj2_dvAVB)
HM_dvAVB_sem = stats.sem(HM_dvAVB); surf_adj2_dvAVB_sem = stats.sem(surf_adj2_dvAVB)


print("dvAVB :", ranksums(HM_dvAVB, surf_adj2_dvAVB)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('mV', fontsize=14, fontname='sans-serif')
ax1.set_title('Basal input AVB', fontsize=18, fontname='sans-serif', y=1.05)
ax1.bar(xpos[0], HM_dvAVB_mean, yerr=HM_dvAVB_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_dvAVB_mean, yerr=surf_adj2_dvAVB_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,0.0078)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [0.007,0.007], lw=0.8, c='k')
ax1.text(1.5, 0.007, "*", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


In [ ]:
# compare dvPVC - idx2

HM_dvPVC = np.array(HM_param_list['Voltage_Change_PVC'])
surf_adj2_dvPVC = np.array(surf_adj2_param_list['Voltage_Change_PVC'])


In [ ]:
# statistical test

HM_dvPVC_mean = np.mean(HM_dvPVC); surf_adj2_dvPVC_mean = np.mean(surf_adj2_dvPVC)
HM_dvPVC_sem = stats.sem(HM_dvPVC); surf_adj2_dvPVC_sem = stats.sem(surf_adj2_dvPVC)


print("dvPVC :", ranksums(HM_dvPVC, surf_adj2_dvPVC)[1])


In [ ]:
# bar graph

b_width = 0.6; fsz=14; lsz=14
xpos = [1,2]


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3,4))

ax1.set_ylabel('mV', fontsize=14, fontname='sans-serif')
ax1.set_title('Basal input PVC', fontsize=18, fontname='sans-serif', y=1.05)
ax1.bar(xpos[0], HM_dvPVC_mean, yerr=HM_dvPVC_sem, width=b_width, color="#7F7F7F", alpha=0.7)
ax1.bar(xpos[1], surf_adj2_dvPVC_mean, yerr=surf_adj2_dvPVC_sem, width=b_width, color="#FF7F0E", alpha=0.7)
ax1.set_xlim(0.4,2.6)
ax1.set_ylim(0,0.0083)
ax1.set_xticks(xpos)
ax1.set_xticklabels(["667","1434"], fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [0.0074,0.0074], lw=0.8, c='k')
ax1.text(1.5, 0.0074, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='Arial')

plt.show()


---